# FedSVLM on MiRAGeNews — experiments

Notebook 1 of 2 for the term paper **"Federated Learning with Fine-Tuned Small Vision-Language Models for Multimodal Fake News Detection"** (NLP course, Quy Nhon University).

**What this notebook does.** Trains Light-MFND (CLIP ViT-B/32 + LoRA + cross-attention fusion; Nguyen, Phan & Le, 2026) on MiRAGeNews centrally and with FedAvg over 5 clients (Dirichlet label skew, α ∈ {100, 0.5, 0.1}), and compares it with local-only training, full fine-tuning, single-modality models, late fusion (**FedSVLM-LF**), FedProx and a TF-IDF + SVM baseline. In-domain test = `test1_nyt_mj`; out-of-domain (OOD) = the four other test sets (OOD scores are their mean).

**Environment.** Google Colab, NVIDIA T4 GPU. Data are read from memory-mapped `.npy` arrays in Google Drive (`MyDrive/NLP_project/fed/npy`), produced by `data_prep/00_MiRAGeNews_preprocessing.ipynb`. Every result is appended to `fed/results.json` (also in `results/results_miragenews.json` of this repository).

**Run order.** Top to bottom. Each experiment has its own cell and its own checkpoint; training cells resume from their checkpoint after an interrupted session.

**Notes on the saved outputs.**
- Experiments ran on 16–20 Nov 2026 across several Colab sessions (GPU quota), so execution counters are cleared.
- Cell order was rearranged so that the notebook reads in logical order (e.g. 28c–28d now come before 29b–29c, which use their checkpoints); code and outputs are unchanged.
- Small diagnostic cells (file checks, `nvidia-smi`, an intermediate Drive flush, a data re-copy) were removed.
- Cell 23 (client split 1): only the partition printout was saved in this copy; the training run finished in another session. Its metrics and per-round log are stored in `results.json` under `FedAvg-LoRA (a=0.5, mu=0.0, split=1, seed=0)`.
- Cell 30 was executed before the experiments in Section 8; cell 31 lists all 45 results.

**Glossary of Vietnamese labels in comments and outputs**

| Vietnamese | English |
|---|---|
| Tập trung | Centralized |
| chỉ ảnh / chỉ chữ | image-only / text-only |
| kết hợp muộn | late fusion |
| tăng cường ảnh | image augmentation |
| ngưỡng | decision threshold |
| cách chia client (split) | client partition |
| vòng | round |
| chú thích / ảnh trùng | duplicated captions / images |
| Nhóm / Phương pháp / Số lần | Group / Method / Number of runs |
| Học liên kết | Federated learning |
| MB/lượt gửi | MB per update |

## 0. Setup
Libraries, data loading (memory-mapped arrays), model and training helpers, shared LoRA initialisation (`init.pt`).

In [ ]:
# Ô 1
!pip install -q peft datasets scikit-learn
!pip install -q -U torchao

In [ ]:
# Ô 2
import os, shutil, numpy as np, torch
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
FED = "/content/drive/MyDrive/NLP_project/fed"; MODEL_NAME = "openai/clip-vit-base-patch32"
NPY = "/content/npy"
if not os.path.exists(NPY): shutil.copytree(f"{FED}/npy", NPY)
SPLITS = ["train", "validation", "test1_nyt_mj", "test2_bbc_dalle", "test3_cnn_dalle", "test4_bbc_sdxl", "test5_cnn_sdxl"]
TESTS = SPLITS[2:]
class MM(torch.utils.data.Dataset):
    def __init__(s, sp):
        s.pix = np.load(f"{NPY}/{sp}_pix.npy", mmap_mode="r")
        s.ids = np.load(f"{NPY}/{sp}_ids.npy"); s.msk = np.load(f"{NPY}/{sp}_msk.npy"); s.labels = np.load(f"{NPY}/{sp}_lab.npy")
    def __len__(s): return len(s.labels)
    def __getitem__(s, i):
        return dict(input_ids=torch.from_numpy(s.ids[i]), attention_mask=torch.from_numpy(s.msk[i]),
                    pixel_values=torch.from_numpy(np.array(s.pix[i])), labels=torch.tensor(s.labels[i]))
enc = {sp: MM(sp) for sp in SPLITS}
print({sp: len(d) for sp, d in enc.items()})

{'train': 10000, 'validation': 2500, 'test1_nyt_mj': 500, 'test2_bbc_dalle': 500, 'test3_cnn_dalle': 500, 'test4_bbc_sdxl': 500, 'test5_cnn_sdxl': 500}


In [ ]:
# Ô 3
import copy, time, json, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from transformers import CLIPModel
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True

class CLIPForMFND(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        base = CLIPModel.from_pretrained(model_name); d = base.config.projection_dim
        self.clip = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"],
                                                    lora_dropout=0.1, bias="none"))
        self.cross_attention = nn.MultiheadAttention(d, 8, batch_first=True)
        self.W_g = nn.Linear(2*d, d); self.classifier = nn.Linear(d, num_labels)
        self.lw, self.delta = lambda_weight, delta
    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        o = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, return_dict=True)
        hT, hI = o.text_embeds, o.image_embeds
        z, _ = self.cross_attention(hT.unsqueeze(1), hI.unsqueeze(1), hI.unsqueeze(1)); z = z.squeeze(1)
        hf = torch.sigmoid(self.W_g(torch.cat([hT, hI], 1))) * z
        logits = self.classifier(hf); loss = None
        if labels is not None:
            sim = F.cosine_similarity(hT, hI)
            la = torch.where(labels == 0, 1 - sim, torch.clamp(sim - self.delta, min=0.0)).mean()
            loss = F.cross_entropy(logits, labels) + self.lw * la
        return loss, logits

def build():
    m = CLIPForMFND().to(device)
    return m
def trainable_keys(m): return [n for n, p in m.named_parameters() if p.requires_grad]
def get_state(m, keys):
    P = dict(m.named_parameters()); return {k: P[k].detach().cpu().clone() for k in keys}
def set_state(m, st):
    P = dict(m.named_parameters())
    with torch.no_grad():
        for k, v in st.items(): P[k].copy_(v.to(P[k].device))
def n_params(m):
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad); return tr, sum(p.numel() for p in m.parameters())

def loader(dset, bs, shuffle, idx=None):
    d = Subset(dset, idx) if idx is not None else dset
    return DataLoader(d, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

def batch_to(b):
    d = {k: b[k].to(device, non_blocking=True) for k in ("input_ids", "attention_mask", "pixel_values", "labels")}
    d["pixel_values"] = d["pixel_values"].float(); return d

def train_epochs(m, dset, idx=None, epochs=1, lr=1e-5, wd=0.01, bs=16, ref=None, mu=0.0):
    m.train(); named = [(n, p) for n, p in m.named_parameters() if p.requires_grad]; ps = [p for _, p in named]
    refs = [ref[n].to(device) for n, _ in named] if (ref is not None and mu > 0) else None
    opt = torch.optim.AdamW(ps, lr=lr, weight_decay=wd); sc = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    tot, n = 0.0, 0
    for _ in range(epochs):
        for b in loader(dset, bs, True, idx):
            b = batch_to(b)
            with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
                loss, _ = m(b["input_ids"], b["attention_mask"], b["pixel_values"], b["labels"])
            if refs is not None:                                   # FedProx: kéo θ cục bộ về θ toàn cục
                loss = loss + (mu / 2) * sum(((p - r_) ** 2).sum() for p, r_ in zip(ps, refs))
            opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.step(opt); sc.update()
            tot += loss.item() * len(b["labels"]); n += len(b["labels"])
    return tot / max(n, 1)

@torch.no_grad()
def evaluate(m, dset, bs=64):
    m.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast("cuda", dtype=torch.float16):
            _, lg = m(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    p, y = torch.cat(P).numpy(), torch.cat(Y).numpy(); pred = (p > 0.5).astype(int)
    return dict(acc=accuracy_score(y, pred)*100, f1=f1_score(y, pred)*100, auc=roc_auc_score(y, p)*100)

def eval_all(m, name):
    r = {"val": evaluate(m, enc["validation"])}
    for s in TESTS: r[s] = evaluate(m, enc[s])
    ood = [r[s] for s in TESTS if s != "test1_nyt_mj"] if "test1_nyt_mj" in TESTS else [r[s] for s in TESTS[1:]]
    r["OOD"] = {k: float(np.mean([x[k] for x in ood])) for k in ("acc","f1","auc")}
    print(f"[{name}] val F1 {r['val']['f1']:.2f} | test1 Acc/F1 {r[TESTS[0]]['acc']:.1f}/{r[TESTS[0]]['f1']:.1f} | "
          f"OOD Acc/F1/AUC {r['OOD']['acc']:.1f}/{r['OOD']['f1']:.1f}/{r['OOD']['auc']:.1f}")
    return r

m = build(); KEYS = trainable_keys(m); tr, tot = n_params(m)
print(f"Trainable {tr:,} / {tot:,} ({tr/tot*100:.2f}%)  — repo báo 2,067,970 / 153,345,283")
bytes_fp32 = lambda ks: sum(v.numel() for v in get_state(m, ks).values()) * 4
UPDATE_MB = bytes_fp32(KEYS)/1e6; FULL_MB = tot*4/1e6
print(f"Mỗi lượt gửi (LoRA+gate+head) ≈ {UPDATE_MB:.2f} MB | toàn bộ mô hình ≈ {FULL_MB:.1f} MB | tỉ lệ ≈ {FULL_MB/UPDATE_MB:.0f}x")
INIT_PATH = f"{FED}/init.pt"
if os.path.exists(INIT_PATH): INIT = torch.load(INIT_PATH); set_state(m, INIT); print("Dùng lại khởi tạo đã lưu")
else: INIT = get_state(m, KEYS); torch.save(INIT, INIT_PATH); print("Tạo và lưu khởi tạo mới")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Trainable 2,067,970 / 153,345,283 (1.35%)  — repo báo 2,067,970 / 153,345,283
Mỗi lượt gửi (LoRA+gate+head) ≈ 8.27 MB | toàn bộ mô hình ≈ 613.4 MB | tỉ lệ ≈ 74x
Dùng lại khởi tạo đã lưu


## 1. Main comparison: centralized vs. FedAvg vs. local-only
Dirichlet partitions (α = 100, 0.5, 0.1), 20 rounds × 1 local epoch, communication cost, summary tables.

In [ ]:
# Ô 4
RES_PATH = f"{FED}/results.json"
def save_result(key, val):
    R = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
    R[key] = val; json.dump(R, open(RES_PATH, "w"), indent=1); return R
set_state(m, torch.load(f"{FED}/cen_best.pt")); _ = eval_all(m, "Centralized (memmap)")
set_state(m, INIT)

[Centralized (memmap)] val F1 99.44 | test1 Acc/F1 100.0/100.0 | OOD Acc/F1/AUC 72.4/70.3/81.7


In [ ]:
# Ô 5
K, ALPHA, SEED = 5, 0.1, 0       # 0.5 = vừa, 0.1 = non-IID mạnh, 100 = gần IID
def dirichlet_split(labels, K, alpha, seed, min_size=100):
    rng = np.random.default_rng(seed); labels = np.asarray(labels)
    while True:
        parts = [[] for _ in range(K)]
        for c in np.unique(labels):
            idx = np.where(labels == c)[0]; rng.shuffle(idx)
            p = rng.dirichlet([alpha] * K); cuts = (np.cumsum(p) * len(idx)).astype(int)[:-1]
            for k, s in enumerate(np.split(idx, cuts)): parts[k] += s.tolist()
        if min(len(p) for p in parts) >= min_size: return parts
ytr = enc["train"].labels
PARTS = dirichlet_split(ytr, K, ALPHA, SEED)
for k, p in enumerate(PARTS): print(f"client {k}: n={len(p):5d} | fake={ytr[p].mean()*100:5.1f}%")

client 0: n= 1608 | fake=  2.7%
client 1: n=  844 | fake=  0.0%
client 2: n= 5953 | fake= 78.1%
client 3: n= 1271 | fake=  0.0%
client 4: n=  324 | fake= 93.8%


In [ ]:
# Ô 6
ROUNDS, LOCAL_EP, PATIENCE, MU = 20, 1, 20, 0.0   # MU=0 là FedAvg thuần; MU=0.1 là FedProx
TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU})"; CK = f"{FED}/ck_a{ALPHA}_mu{MU}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, bad, best_g, LOG, r0 = S["g"], float(S["best"]), S["bad"], S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1} (bad={bad})")
else:
    g = copy.deepcopy(INIT); best, bad, best_g, LOG, r0 = -1.0, 0, None, [], 0
torch.manual_seed(r0)
for r in range(r0 + 1, ROUNDS + 1):
    if bad >= PATIENCE: print("đã early stop từ trước"); break
    t0 = time.time(); acc_state = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}
    N = sum(len(p) for p in PARTS); losses = []
    for k in range(K):
        set_state(m, g)
        losses.append(train_epochs(m, enc["train"], PARTS[k], epochs=LOCAL_EP, ref=g, mu=MU))
        st = get_state(m, KEYS)
        for key in acc_state: acc_state[key] += st[key] * (len(PARTS[k]) / N)
    g = acc_state
    set_state(m, g); v = evaluate(m, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, bad, best_g = float(v["auc"]), 0, copy.deepcopy(g)
    else: bad += 1
    torch.save(dict(g=g, best=float(best), bad=int(bad), best_g=best_g, LOG=LOG, r=int(r)), CK)
    if bad >= PATIENCE: print("early stop"); break
set_state(m, best_g); res = eval_all(m, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG)
RESULTS = save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{TAG.replace(' ','_')}_best.pt")

Tiếp tục từ vòng 21 (bad=1)
[FedAvg-LoRA (a=0.1, mu=0.0)] val F1 99.07 | test1 Acc/F1 99.6/99.6 | OOD Acc/F1/AUC 68.8/56.4/83.6


In [ ]:
# Ô 7
LOC_EP = 5; loc = []
for k in range(K):
    set_state(m, INIT); torch.manual_seed(k)
    train_epochs(m, enc["train"], PARTS[k], epochs=LOC_EP)
    loc.append(eval_all(m, f"Local client {k}"))
avg = lambda f: float(np.mean([f(r) for r in loc]))
res = {"val": {"f1": avg(lambda r: r["val"]["f1"])},
       **{s: {k_: avg(lambda r, s=s, k_=k_: r[s][k_]) for k_ in ("acc","f1","auc")} for s in TESTS},
       "OOD": {k_: avg(lambda r, k_=k_: r["OOD"][k_]) for k_ in ("acc","f1","auc")}, "clients": loc}
RESULTS = save_result(f"Local-only (a={ALPHA})", res)

[Local client 0] val F1 0.00 | test1 Acc/F1 50.0/0.0 | OOD Acc/F1/AUC 50.0/0.0/62.1
[Local client 1] val F1 0.00 | test1 Acc/F1 50.0/0.0 | OOD Acc/F1/AUC 50.0/0.0/51.0
[Local client 2] val F1 99.08 | test1 Acc/F1 99.6/99.6 | OOD Acc/F1/AUC 74.1/71.5/83.6
[Local client 3] val F1 0.00 | test1 Acc/F1 50.0/0.0 | OOD Acc/F1/AUC 50.0/0.0/58.5
[Local client 4] val F1 66.67 | test1 Acc/F1 50.0/66.7 | OOD Acc/F1/AUC 50.0/66.7/56.8


In [ ]:
# Ô 8
import pandas as pd
R = json.load(open(f"{FED}/results.json"))
UPD, FULL, K = 2067970*4/1e6, 153345283*4/1e6, 5
rows = []
for name, r in R.items():
    row = {"Phương pháp": name, "val F1": r["val"]["f1"]}
    for s in ["test1_nyt_mj","test2_bbc_dalle","test3_cnn_dalle","test4_bbc_sdxl","test5_cnn_sdxl"]:
        row[s[:5]] = f"{r[s]['acc']:.1f}/{r[s]['f1']:.1f}"
    row["OOD Acc"], row["OOD F1"], row["OOD AUC"] = r["OOD"]["acc"], r["OOD"]["f1"], r["OOD"]["auc"]
    T = r.get("rounds_run"); row["Vòng"] = T
    row["Truyền thông LoRA (GB)"] = round(2*T*K*UPD/1e3, 2) if T else None
    rows.append(row)
print(pd.DataFrame(rows).round(2).to_string(index=False))
print(f"\nGửi toàn bộ mô hình 20 vòng ≈ {2*20*K*FULL/1e3:.1f} GB (ước tính từ kích thước, chưa chạy thật)")

                Phương pháp  val F1       test1     test2     test3     test4     test5  OOD Acc  OOD F1  OOD AUC  Vòng  Truyền thông LoRA (GB)
        FedAvg-LoRA (a=0.5)   74.08   76.6/71.3 60.4/68.7 65.4/72.1 58.8/67.5 62.0/67.7    61.65   68.98    70.06   6.0                    0.50
         Local-only (a=0.5)   64.50   69.1/65.8 55.9/55.4 58.3/57.1 56.2/55.9 59.2/57.8    57.40   56.54    66.40   NaN                     NaN
         Centralized (LoRA)   99.44 100.0/100.0 69.4/66.1 64.6/59.1 77.8/78.2 77.8/77.8    72.40   70.29    81.67   NaN                     NaN
FedAvg-LoRA (a=100, mu=0.0)   99.32   99.8/99.8 63.0/58.0 60.8/49.7 72.6/72.9 78.6/77.7    68.75   64.58    78.69  20.0                    1.65
FedAvg-LoRA (a=0.5, mu=0.0)   99.08   99.6/99.6 67.0/57.4 62.2/45.8 78.0/75.6 81.2/78.8    72.10   64.40    86.52  20.0                    1.65
FedAvg-LoRA (a=0.1, mu=0.0)   99.07   99.6/99.6 56.6/35.6 55.4/30.1 80.2/78.8 82.8/81.1    68.75   56.41    83.63  20.0                 

In [ ]:
# Ô 9
import json
P = f"{FED}/results.json"; R = json.load(open(P))
if "FedAvg-LoRA (a=0.5)" in R:
    R["[PHỤ] FedAvg a=0.5, dừng sớm ở vòng 5 (patience 3, chọn theo F1)"] = R.pop("FedAvg-LoRA (a=0.5)")
json.dump(R, open(P, "w"), indent=1); print(list(R))

['Local-only (a=0.5)', 'Centralized (LoRA)', 'FedAvg-LoRA (a=100, mu=0.0)', 'FedAvg-LoRA (a=0.5, mu=0.0)', 'FedAvg-LoRA (a=0.1, mu=0.0)', 'Local-only (a=0.1)', '[PHỤ] FedAvg a=0.5, dừng sớm ở vòng 5 (patience 3, chọn theo F1)']


In [ ]:
# Ô 10
import pandas as pd
@torch.no_grad()
def probs(m, dset, bs=64):
    m.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            _, lg = m(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

CKPTS = {"Tập trung": f"{FED}/cen_best.pt",
         "FedAvg IID (a=100)": f"{FED}/fed_FedAvg-LoRA_(a=100,_mu=0.0)_best.pt",
         "FedAvg a=0.5": f"{FED}/fed_FedAvg-LoRA_(a=0.5,_mu=0.0)_best.pt",
         "FedAvg a=0.1": f"{FED}/fed_FedAvg-LoRA_(a=0.1,_mu=0.0)_best.pt"}
rows = []
for name, path in CKPTS.items():
    if not os.path.exists(path): print("thiếu file:", path); continue
    set_state(m, torch.load(path)); out = {"Phương pháp": name}
    for tag, splits in [("test1", TESTS[:1]), ("OOD", TESTS[1:])]:
        pr = [probs(m, enc[s]) for s in splits]
        p = np.concatenate([x[0] for x in pr]); y = np.concatenate([x[1] for x in pr]); pred = p > 0.5
        out[f"{tag}: bắt được fake (%)"] = round(100 * pred[y == 1].mean(), 1)
        out[f"{tag}: bắt được real (%)"] = round(100 * (~pred[y == 0]).mean(), 1)
    rows.append(out)
print(pd.DataFrame(rows).to_string(index=False))

       Phương pháp  test1: bắt được fake (%)  test1: bắt được real (%)  OOD: bắt được fake (%)  OOD: bắt được real (%)
         Tập trung                     100.0                     100.0                    67.0                    77.8
FedAvg IID (a=100)                     100.0                      99.6                    59.5                    78.0
      FedAvg a=0.5                      99.6                      99.6                    53.6                    90.6
      FedAvg a=0.1                      99.2                     100.0                    47.7                    89.8


## 2. Full fine-tuning under FedAvg
Same architecture without LoRA (all ~153 M parameters exchanged).

In [ ]:
# Ô 11 — dựng mô hình full fine-tune
import gc, torch, torch.nn as nn
from transformers import CLIPModel

# Kiểm tra các ô trước đã chạy chưa
for name in ["MODEL_NAME", "CLIPForMFND", "device", "n_params", "set_state", "train_epochs",
             "evaluate", "eval_all", "save_result", "dirichlet_split", "enc", "FED"]:
    assert name in globals(), f"Thiếu '{name}': hãy chạy lại ô 1, 2, 3, 4, 5 trước ô 11"

class CLIPFullMFND(nn.Module):   # cùng kiến trúc Light-MFND, KHÔNG dùng LoRA: mọi tham số đều được huấn luyện
    def __init__(self, model_name=MODEL_NAME, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name); d = self.clip.config.projection_dim
        self.cross_attention = nn.MultiheadAttention(d, 8, batch_first=True)
        self.W_g = nn.Linear(2 * d, d); self.classifier = nn.Linear(d, num_labels)
        self.lw, self.delta = lambda_weight, delta
    forward = CLIPForMFND.forward

# Giải phóng mô hình LoRA của ô 3 (ô 13 không cần) để đỡ tốn bộ nhớ GPU
if "m" in globals():
    del m
gc.collect(); torch.cuda.empty_cache()

torch.manual_seed(0)
mf = CLIPFullMFND().to(device)
KEYS_F = [n for n, p in mf.named_parameters() if p.requires_grad]
trf, totf = n_params(mf)
print(f"Full FT: trainable {trf:,} / {totf:,} ({trf/totf*100:.1f}%)  — kỳ vọng 152,853,763")

LORA_MB = globals().get("UPDATE_MB", 2067970 * 4 / 1e6)   # 8.27 MB nếu ô 3 chưa lưu biến này
FULL_UPDATE_MB = trf * 4 / 1e6
print(f"Mỗi lượt gửi ≈ {FULL_UPDATE_MB:.1f} MB (LoRA {LORA_MB:.2f} MB, gấp ≈ {FULL_UPDATE_MB/LORA_MB:.0f} lần)")

def get_state_fast(m, keys):
    P = dict(m.named_parameters()); return {k: P[k].detach().cpu() for k in keys}

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Full FT: trainable 152,853,763 / 152,853,763 (100.0%)  — kỳ vọng 152,853,763
Mỗi lượt gửi ≈ 611.4 MB (LoRA 8.27 MB, gấp ≈ 74 lần)


In [ ]:
# Ô 12
ALPHA, K, SEED = 0.5, 5, 0
ytr = enc["train"].labels
PARTS = dirichlet_split(ytr, K, ALPHA, SEED)
for k, p in enumerate(PARTS): print(f"client {k}: n={len(p):5d} | fake={ytr[p].mean()*100:5.1f}%")

client 0: n= 3777 | fake=  0.8%
client 1: n= 1699 | fake= 64.0%
client 2: n= 3414 | fake= 83.6%
client 3: n=  290 | fake= 85.2%
client 4: n=  820 | fake= 95.2%


In [ ]:
# Ô 13
ROUNDS, LOCAL_EP, SAVE_EVERY = 20, 1, 5
TAG = f"FedAvg-FullFT (a={ALPHA})"; CKF = f"{FED}/ckfull_a{ALPHA}.pt"
if "FS" in globals() and FS.get("tag") == TAG:
    print(f"Tiếp tục từ bộ nhớ, vòng {FS['r']+1}")
elif os.path.exists(CKF):
    FS = torch.load(CKF, weights_only=False); print(f"Tiếp tục từ checkpoint, vòng {FS['r']+1}")
else:
    FS = dict(tag=TAG, g=get_state_fast(mf, KEYS_F), best=-1.0, bad=0, best_g=None, LOG=[], r=0)
torch.cuda.reset_peak_memory_stats(); torch.manual_seed(FS["r"])
for r in range(FS["r"] + 1, ROUNDS + 1):
    t0 = time.time(); t_tr = 0.0; t_st = 0.0
    g = FS["g"]; acc = {k: torch.zeros_like(v) for k, v in g.items()}; N = sum(len(p) for p in PARTS); losses = []
    for k in range(K):
        a = time.time(); set_state(mf, g); t_st += time.time() - a
        a = time.time(); losses.append(train_epochs(mf, enc["train"], PARTS[k], epochs=LOCAL_EP)); t_tr += time.time() - a
        a = time.time(); st = get_state_fast(mf, KEYS_F); w = len(PARTS[k]) / N
        for key in acc: acc[key] += st[key] * w
        t_st += time.time() - a
    g = acc; set_state(mf, g); v = evaluate(mf, enc["validation"])
    FS["g"] = g; FS["r"] = r
    FS["LOG"].append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                          loss=float(np.mean(losses)), sec=float(time.time() - t0), sec_train=float(t_tr), sec_state=float(t_st)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | "
          f"tổng {time.time()-t0:.0f}s (train {t_tr:.0f}s, chuyển/gộp tham số {t_st:.0f}s) | GPU đỉnh {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
    if v["auc"] > FS["best"]: FS["best"], FS["bad"], FS["best_g"] = float(v["auc"]), 0, {k_: t_.clone() for k_, t_ in g.items()}
    else: FS["bad"] += 1
    if r % SAVE_EVERY == 0: torch.save(FS, CKF); print("  đã lưu checkpoint lên Drive")
if FS["r"] >= 20:
    set_state(mf, FS["best_g"]); res = eval_all(mf, TAG)
    res.update(rounds_run=len(FS["LOG"]), best_round=int(np.argmax([x["val_auc"] for x in FS["LOG"]])) + 1,
               time_s=float(sum(x["sec"] for x in FS["LOG"])), update_mb=float(FULL_UPDATE_MB), log=FS["LOG"])
    RESULTS = save_result(TAG, res); torch.save(FS, CKF); print("Đã lưu kết quả full fine-tune vào results.json")

Tiếp tục từ checkpoint, vòng 6
round  6 | loss 0.2734 | val Acc 80.48 F1 75.77 AUC 99.24 | tổng 123s (train 113s, chuyển/gộp tham số 5s) | GPU đỉnh 4.2 GB
round  7 | loss 0.1537 | val Acc 64.04 F1 43.85 AUC 98.92 | tổng 116s (train 107s, chuyển/gộp tham số 4s) | GPU đỉnh 4.2 GB
round  8 | loss 0.2451 | val Acc 74.00 F1 64.86 AUC 99.08 | tổng 116s (train 107s, chuyển/gộp tham số 4s) | GPU đỉnh 4.2 GB
round  9 | loss 0.1660 | val Acc 84.12 F1 81.14 AUC 99.33 | tổng 117s (train 109s, chuyển/gộp tham số 4s) | GPU đỉnh 4.2 GB
round 10 | loss 0.0871 | val Acc 91.28 F1 90.51 AUC 99.44 | tổng 117s (train 108s, chuyển/gộp tham số 4s) | GPU đỉnh 4.2 GB
  đã lưu checkpoint lên Drive
round 11 | loss 0.0843 | val Acc 95.84 F1 95.85 AUC 99.25 | tổng 136s (train 126s, chuyển/gộp tham số 5s) | GPU đỉnh 4.2 GB
round 12 | loss 0.0799 | val Acc 95.32 F1 95.16 AUC 99.41 | tổng 116s (train 107s, chuyển/gộp tham số 4s) | GPU đỉnh 4.2 GB
round 13 | loss 0.0593 | val Acc 95.20 F1 95.05 AUC 99.43 | tổng 117s (

## 3. Decision-threshold analysis on OOD data
No retraining; thresholds chosen on three OOD sets and tested on the fourth.

In [ ]:
# Ô 14 — Chọn lại ngưỡng trên OOD (chỉ để phân tích, không huấn luyện lại)
import gc, os, numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, roc_auc_score

for n in ["build", "set_state", "enc", "TESTS", "FED", "loader", "batch_to", "device"]:
    assert n in globals(), f"Thiếu '{n}': hãy chạy ô 1, 2, 3 trước"

if "mf" in globals(): del mf          # giải phóng mô hình full fine-tune nếu còn
gc.collect(); torch.cuda.empty_cache()
if "m" not in globals(): m = build()   # mô hình LoRA

@torch.no_grad()
def probs(m, dset, bs=64):
    m.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            _, lg = m(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

CKPTS = {"Tập trung": f"{FED}/cen_best.pt",
         "FedAvg IID (a=100)": f"{FED}/fed_FedAvg-LoRA_(a=100,_mu=0.0)_best.pt",
         "FedAvg a=0.5": f"{FED}/fed_FedAvg-LoRA_(a=0.5,_mu=0.0)_best.pt",
         "FedAvg a=0.1": f"{FED}/fed_FedAvg-LoRA_(a=0.1,_mu=0.0)_best.pt"}
OOD = TESTS[1:]; ts = np.linspace(0.01, 0.99, 99)
rows = []
for name, path in CKPTS.items():
    if not os.path.exists(path): print("thiếu file:", path); continue
    set_state(m, torch.load(path))
    P = {s: probs(m, enc[s]) for s in OOD}
    f1s = lambda t, ss: np.mean([f1_score(P[s][1], P[s][0] > t) * 100 for s in ss])
    f_05 = f1s(0.5, OOD)                                   # phải khớp bảng ô 8
    curve = [f1s(t, OOD) for t in ts]; t_or = ts[int(np.argmax(curve))]
    loo = []                                               # chọn ngưỡng trên 3 tập, đánh giá trên tập còn lại
    for s in OOD:
        others = [x for x in OOD if x != s]
        t_ = ts[int(np.argmax([f1s(t, others) for t in ts]))]
        loo.append(f1_score(P[s][1], P[s][0] > t_) * 100)
    auc = np.mean([roc_auc_score(P[s][1], P[s][0]) * 100 for s in OOD])
    rows.append({"Phương pháp": name, "OOD F1 @0.5": round(f_05, 2),
                 "OOD F1 (ngưỡng tốt nhất)": round(max(curve), 2), "ngưỡng đó": round(t_or, 2),
                 "OOD F1 (chọn trên 3 tập, thử tập còn lại)": round(float(np.mean(loo)), 2),
                 "OOD AUC": round(auc, 2)})
print(pd.DataFrame(rows).to_string(index=False))

       Phương pháp  OOD F1 @0.5  OOD F1 (ngưỡng tốt nhất)  ngưỡng đó  OOD F1 (chọn trên 3 tập, thử tập còn lại)  OOD AUC
         Tập trung        70.29                     77.49       0.01                                      77.05    81.67
FedAvg IID (a=100)        64.58                     77.03       0.03                                      76.15    78.69
      FedAvg a=0.5        64.40                     80.13       0.01                                      79.50    86.52
      FedAvg a=0.1        56.41                     73.18       0.01                                      73.18    83.63


## 4. More seeds and a budget-matched centralized baseline

In [ ]:
# Ô 15 — thêm seed cho FedAvg-LoRA (a=0.5)
import copy, time, os, torch, numpy as np
for n in ["build","set_state","get_state","train_epochs","evaluate","eval_all","save_result",
          "dirichlet_split","enc","FED","INIT","KEYS"]:
    assert n in globals(), f"Thiếu '{n}': chạy lại ô 1, 2, 3, 4 trước"
if "m" not in globals(): m = build()

SEED_RUN = 1                      # lần sau đổi thành 2
ALPHA, K, ROUNDS, LOCAL_EP, MU = 0.5, 5, 20, 1, 0.0
PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, 0)      # giống cách chia đã dùng
TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU}, seed={SEED_RUN})"
CK = f"{FED}/ck_a{ALPHA}_mu{MU}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = copy.deepcopy(INIT); best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m, g)
        losses.append(train_epochs(m, enc["train"], PARTS[k], epochs=LOCAL_EP, ref=g, mu=MU))
        st = get_state(m, KEYS)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m, g); v = evaluate(m, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), bad=0, best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m, best_g); res = eval_all(m, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{TAG.replace(' ','_')}_best.pt")

Tiếp tục từ vòng 10
round 10 | loss 0.0677 | val Acc 97.84 F1 97.83 AUC 99.75 | 82s
round 11 | loss 0.0536 | val Acc 97.72 F1 97.68 AUC 99.80 | 74s
round 12 | loss 0.0469 | val Acc 98.16 F1 98.15 AUC 99.83 | 74s
round 13 | loss 0.0458 | val Acc 98.16 F1 98.13 AUC 99.84 | 72s
round 14 | loss 0.0423 | val Acc 98.24 F1 98.22 AUC 99.86 | 70s
round 15 | loss 0.0368 | val Acc 98.36 F1 98.34 AUC 99.88 | 69s
round 16 | loss 0.0331 | val Acc 98.72 F1 98.71 AUC 99.89 | 69s
round 17 | loss 0.0327 | val Acc 98.96 F1 98.95 AUC 99.90 | 69s
round 18 | loss 0.0314 | val Acc 98.64 F1 98.62 AUC 99.90 | 68s
round 19 | loss 0.0301 | val Acc 98.92 F1 98.91 AUC 99.90 | 69s
round 20 | loss 0.0256 | val Acc 99.04 F1 99.04 AUC 99.91 | 69s
[FedAvg-LoRA (a=0.5, mu=0.0, seed=1)] val F1 99.04 | test1 Acc/F1 99.8/99.8 | OOD Acc/F1/AUC 75.2/70.4/85.9


In [ ]:
# Ô 16
import json, numpy as np
R = json.load(open(f"{FED}/results.json"))
keys = ["FedAvg-LoRA (a=0.5, mu=0.0)"] + [f"FedAvg-LoRA (a=0.5, mu=0.0, seed={s})" for s in (1, 2)]
keys = [k for k in keys if k in R]; print("Số lần chạy:", len(keys))
for name, f in [("test1 F1", lambda r: r["test1_nyt_mj"]["f1"]), ("OOD Acc", lambda r: r["OOD"]["acc"]),
                ("OOD F1", lambda r: r["OOD"]["f1"]), ("OOD AUC", lambda r: r["OOD"]["auc"])]:
    v = [f(R[k]) for k in keys]; print(f"{name:8s}: {np.mean(v):.2f} ± {np.std(v, ddof=1) if len(v)>1 else float('nan'):.2f}  {[round(x,2) for x in v]}")

Số lần chạy: 2
test1 F1: 99.70 ± 0.14  [99.6, 99.8]
OOD Acc : 73.62 ± 2.16  [72.1, 75.15]
OOD F1  : 67.40 ± 4.24  [64.4, 70.39]
OOD AUC : 86.23 ± 0.41  [86.52, 85.94]


In [ ]:
# Ô 17 — thêm seed cho FedAvg-LoRA (a=0.5)
import copy, time, os, torch, numpy as np
for n in ["build","set_state","get_state","train_epochs","evaluate","eval_all","save_result",
          "dirichlet_split","enc","FED","INIT","KEYS"]:
    assert n in globals(), f"Thiếu '{n}': chạy lại ô 1, 2, 3, 4, 5 trước"
if "m" not in globals(): m = build()

SEED_RUN = 2
ALPHA, K, ROUNDS, LOCAL_EP, MU = 0.5, 5, 20, 1, 0.0
PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, 0)      # giống cách chia đã dùng
TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU}, seed={SEED_RUN})"
CK = f"{FED}/ck_a{ALPHA}_mu{MU}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = copy.deepcopy(INIT); best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m, g)
        losses.append(train_epochs(m, enc["train"], PARTS[k], epochs=LOCAL_EP, ref=g, mu=MU))
        st = get_state(m, KEYS)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m, g); v = evaluate(m, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), bad=0, best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m, best_g); res = eval_all(m, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{TAG.replace(' ','_')}_best.pt")

round  1 | loss 0.6882 | val Acc 50.84 F1 3.30 AUC 81.92 | 82s
round  2 | loss 0.6579 | val Acc 58.24 F1 28.79 AUC 92.38 | 68s
round  3 | loss 0.6098 | val Acc 62.88 F1 41.19 AUC 96.47 | 68s
round  4 | loss 0.5423 | val Acc 63.84 F1 43.50 AUC 97.88 | 68s
round  5 | loss 0.4596 | val Acc 63.12 F1 41.65 AUC 98.37 | 68s
round  6 | loss 0.3716 | val Acc 74.84 F1 66.45 AUC 98.87 | 67s
round  7 | loss 0.2668 | val Acc 93.72 F1 93.40 AUC 99.30 | 68s
round  8 | loss 0.1519 | val Acc 97.00 F1 96.98 AUC 99.53 | 68s
round  9 | loss 0.0950 | val Acc 97.44 F1 97.42 AUC 99.68 | 68s
round 10 | loss 0.0659 | val Acc 97.88 F1 97.87 AUC 99.75 | 68s
round 11 | loss 0.0570 | val Acc 98.00 F1 97.98 AUC 99.80 | 68s
round 12 | loss 0.0499 | val Acc 98.08 F1 98.06 AUC 99.83 | 68s
round 13 | loss 0.0455 | val Acc 98.32 F1 98.30 AUC 99.85 | 68s
round 14 | loss 0.0446 | val Acc 98.08 F1 98.05 AUC 99.86 | 68s
round 15 | loss 0.0376 | val Acc 98.72 F1 98.72 AUC 99.87 | 69s
round 16 | loss 0.0339 | val Acc 98.88 F1

In [ ]:
# Ô 18 — Tập trung, thêm seed
import time, torch, numpy as np
SEED_RUN = 2                     # chạy xong đổi thành 2
TAG = f"Centralized (LoRA, seed={SEED_RUN})"
set_state(m, INIT); torch.manual_seed(1000 * SEED_RUN)
best, bad, best_state, t0 = -1.0, 0, None, time.time()
for ep in range(1, 6):
    l = train_epochs(m, enc["train"], epochs=1)
    v = evaluate(m, enc["validation"])
    print(f"epoch {ep} loss {l:.4f} | val F1 {v['f1']:.2f} AUC {v['auc']:.2f}")
    if v["f1"] > best: best, bad, best_state = v["f1"], 0, get_state(m, KEYS)
    else:
        bad += 1
        if bad >= 2: break
set_state(m, best_state); res = eval_all(m, TAG); res["time_s"] = time.time() - t0; res["epochs_run"] = ep
save_result(TAG, res); torch.save(best_state, f"{FED}/cen_s{SEED_RUN}_best.pt")

epoch 1 loss 0.5778 | val F1 98.55 AUC 99.87
epoch 2 loss 0.0780 | val F1 99.00 AUC 99.95
epoch 3 loss 0.0415 | val F1 99.13 AUC 99.97
epoch 4 loss 0.0301 | val F1 99.32 AUC 99.98
epoch 5 loss 0.0221 | val F1 99.36 AUC 99.98
[Centralized (LoRA, seed=2)] val F1 99.36 | test1 Acc/F1 100.0/100.0 | OOD Acc/F1/AUC 69.8/65.9/79.5


In [ ]:
# Ô 19 — FedAvg-LoRA, thêm seed cho mọi alpha
import copy, time, os, torch, numpy as np
for n in ["build","set_state","get_state","train_epochs","evaluate","eval_all","save_result",
          "dirichlet_split","enc","FED","INIT","KEYS"]:
    assert n in globals(), f"Thiếu '{n}': chạy lại ô 1, 2, 3, 4, 5 trước"
if "m" not in globals(): m = build()

ALPHA, SEED_RUN = 0.1, 2          # đổi ở đây
K, ROUNDS, LOCAL_EP, MU = 5, 20, 1, 0.0
PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, 0)      # giống cách chia đã dùng
TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU}, seed={SEED_RUN})"
CK = f"{FED}/ck_a{ALPHA}_mu{MU}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = copy.deepcopy(INIT); best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m, g)
        losses.append(train_epochs(m, enc["train"], PARTS[k], epochs=LOCAL_EP, ref=g, mu=MU))
        st = get_state(m, KEYS)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m, g); v = evaluate(m, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), bad=0, best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m, best_g); res = eval_all(m, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{TAG.replace(' ','_')}_best.pt")

round  1 | loss 0.7041 | val Acc 50.00 F1 66.67 AUC 95.09 | 68s
round  2 | loss 0.7027 | val Acc 50.00 F1 66.67 AUC 98.64 | 67s
round  3 | loss 0.6468 | val Acc 77.88 F1 81.87 AUC 99.21 | 67s
round  4 | loss 0.5081 | val Acc 96.04 F1 95.99 AUC 99.23 | 67s
round  5 | loss 0.3118 | val Acc 97.32 F1 97.27 AUC 99.53 | 66s
round  6 | loss 0.1433 | val Acc 98.16 F1 98.14 AUC 99.73 | 67s
round  7 | loss 0.0784 | val Acc 98.16 F1 98.14 AUC 99.81 | 68s
round  8 | loss 0.0541 | val Acc 98.48 F1 98.47 AUC 99.86 | 67s
round  9 | loss 0.0494 | val Acc 98.88 F1 98.87 AUC 99.89 | 67s
round 10 | loss 0.0341 | val Acc 99.00 F1 98.99 AUC 99.91 | 67s
round 11 | loss 0.0335 | val Acc 98.76 F1 98.75 AUC 99.91 | 66s
round 12 | loss 0.0323 | val Acc 99.16 F1 99.16 AUC 99.93 | 66s
round 13 | loss 0.0253 | val Acc 99.12 F1 99.12 AUC 99.94 | 67s
round 14 | loss 0.0224 | val Acc 99.00 F1 98.99 AUC 99.95 | 67s
round 15 | loss 0.0205 | val Acc 99.24 F1 99.24 AUC 99.95 | 67s
round 16 | loss 0.0178 | val Acc 99.24 F

In [ ]:
# Ô 20
import json, numpy as np, pandas as pd
R = json.load(open(f"{FED}/results.json"))
GROUPS = {"Tập trung":      ["Centralized (LoRA)"] + [f"Centralized (LoRA, seed={s})" for s in (1, 2)],
          "FedAvg a=100":   ["FedAvg-LoRA (a=100, mu=0.0)"] + [f"FedAvg-LoRA (a=100, mu=0.0, seed={s})" for s in (1, 2)],
          "FedAvg a=0.5":   ["FedAvg-LoRA (a=0.5, mu=0.0)"] + [f"FedAvg-LoRA (a=0.5, mu=0.0, seed={s})" for s in (1, 2)],
          "FedAvg a=0.1":   ["FedAvg-LoRA (a=0.1, mu=0.0)"] + [f"FedAvg-LoRA (a=0.1, mu=0.0, seed={s})" for s in (1, 2)]}
MET = {"test1 F1": lambda r: r["test1_nyt_mj"]["f1"], "OOD Acc": lambda r: r["OOD"]["acc"],
       "OOD F1": lambda r: r["OOD"]["f1"], "OOD AUC": lambda r: r["OOD"]["auc"]}
rows = []
for g, keys in GROUPS.items():
    keys = [k for k in keys if k in R]; row = {"Phương pháp": g, "Số lần": len(keys)}
    for name, f in MET.items():
        v = [f(R[k]) for k in keys]
        row[name] = f"{np.mean(v):.2f} ± {np.std(v, ddof=1):.2f}" if len(v) > 1 else f"{v[0]:.2f}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

 Phương pháp  Số lần      test1 F1      OOD Acc       OOD F1      OOD AUC
   Tập trung       3 100.00 ± 0.00 71.33 ± 1.36 68.11 ± 2.22 80.61 ± 1.09
FedAvg a=100       1         99.80        68.75        64.58        78.69
FedAvg a=0.5       3  99.67 ± 0.12 74.12 ± 1.75 68.22 ± 3.32 86.05 ± 0.43
FedAvg a=0.1       3  99.73 ± 0.12 69.25 ± 1.14 57.97 ± 2.58 83.55 ± 0.71


In [ ]:
# Ô 21 — Tập trung, cùng ngân sách và cách chọn mô hình như FedAvg (20 epoch, chọn theo val AUC)
import copy, time, os, torch, numpy as np
SEED_RUN = 0                      # chạy 0 trước; nếu còn GPU thì chạy 1
EPOCHS = 20
TAG = f"Centralized (LoRA, 20ep, chọn AUC, seed={SEED_RUN})"
CK = f"{FED}/ck_cen20_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    st, best, best_state, LOG, e0 = S["st"], float(S["best"]), S["best_state"], S["LOG"], S["e"]
    set_state(m, st); print(f"Tiếp tục từ epoch {e0+1}")
else:
    set_state(m, INIT); best, best_state, LOG, e0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + e0)
for ep in range(e0 + 1, EPOCHS + 1):
    t0 = time.time(); l = train_epochs(m, enc["train"], epochs=1); v = evaluate(m, enc["validation"])
    LOG.append(dict(epoch=ep, val_f1=float(v["f1"]), val_auc=float(v["auc"]), loss=float(l), sec=float(time.time() - t0)))
    print(f"epoch {ep:2d} | loss {l:.4f} | val F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_state = float(v["auc"]), get_state(m, KEYS)
    torch.save(dict(st=get_state(m, KEYS), best=best, best_state=best_state, LOG=LOG, e=ep), CK)
set_state(m, best_state); res = eval_all(m, TAG)
res.update(epochs_run=len(LOG), best_epoch=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG)
save_result(TAG, res); torch.save(best_state, f"{FED}/cen20_s{SEED_RUN}_best.pt")

epoch  1 | loss 0.5738 | val F1 98.72 AUC 99.87 | 77s
epoch  2 | loss 0.0783 | val F1 98.89 AUC 99.95 | 67s
epoch  3 | loss 0.0432 | val F1 99.40 AUC 99.97 | 66s
epoch  4 | loss 0.0316 | val F1 99.40 AUC 99.98 | 66s
epoch  5 | loss 0.0250 | val F1 99.24 AUC 99.98 | 67s
epoch  6 | loss 0.0185 | val F1 99.44 AUC 99.98 | 67s
epoch  7 | loss 0.0154 | val F1 99.40 AUC 99.98 | 67s
epoch  8 | loss 0.0112 | val F1 99.44 AUC 99.99 | 66s
epoch  9 | loss 0.0099 | val F1 99.56 AUC 99.98 | 66s
epoch 10 | loss 0.0067 | val F1 99.40 AUC 99.98 | 67s
epoch 11 | loss 0.0065 | val F1 99.48 AUC 99.99 | 67s
epoch 12 | loss 0.0042 | val F1 99.52 AUC 99.98 | 67s
epoch 13 | loss 0.0043 | val F1 99.56 AUC 99.99 | 65s
epoch 14 | loss 0.0043 | val F1 99.56 AUC 99.98 | 66s
epoch 15 | loss 0.0027 | val F1 99.48 AUC 99.99 | 67s
epoch 16 | loss 0.0020 | val F1 99.56 AUC 99.99 | 67s
epoch 17 | loss 0.0052 | val F1 99.56 AUC 99.95 | 67s
epoch 18 | loss 0.0021 | val F1 99.36 AUC 99.79 | 66s
epoch 19 | loss 0.0013 | val

In [ ]:
# Ô 22 — FedAvg-LoRA α=100, thêm seed
import copy, time, os, torch, numpy as np
for n in ["build","set_state","get_state","train_epochs","evaluate","eval_all","save_result",
          "dirichlet_split","enc","FED","INIT","KEYS"]:
    assert n in globals(), f"Thiếu '{n}': chạy lại ô 1, 2, 3, 4, 5 trước"
if "m" not in globals(): m = build()

ALPHA, SEED_RUN = 100, 2         # chạy xong đổi thành 100, 2
K, ROUNDS, LOCAL_EP, MU = 5, 20, 1, 0.0
PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, 0)      # giống cách chia đã dùng
for k, p in enumerate(PARTS): print(f"client {k}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")
TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU}, seed={SEED_RUN})"
CK = f"{FED}/ck_a{ALPHA}_mu{MU}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = copy.deepcopy(INIT); best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m, g)
        losses.append(train_epochs(m, enc["train"], PARTS[k], epochs=LOCAL_EP, ref=g, mu=MU))
        st = get_state(m, KEYS)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m, g); v = evaluate(m, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), bad=0, best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m, best_g); res = eval_all(m, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{TAG.replace(' ','_')}_best.pt")

client 0: n= 2091 | fake= 50.0%
client 1: n= 2133 | fake= 50.7%
client 2: n= 1903 | fake= 46.0%
client 3: n= 2062 | fake= 54.4%
client 4: n= 1811 | fake= 48.3%
round  1 | loss 0.7260 | val Acc 59.40 F1 31.83 AUC 89.30 | 87s
round  2 | loss 0.7208 | val Acc 89.12 F1 88.72 AUC 96.32 | 77s
round  3 | loss 0.7085 | val Acc 90.92 F1 91.49 AUC 98.15 | 78s
round  4 | loss 0.6745 | val Acc 91.60 F1 92.15 AUC 98.95 | 78s
round  5 | loss 0.5948 | val Acc 93.96 F1 94.21 AUC 99.39 | 75s
round  6 | loss 0.4522 | val Acc 97.00 F1 97.02 AUC 99.64 | 75s
round  7 | loss 0.2829 | val Acc 97.68 F1 97.67 AUC 99.76 | 75s
round  8 | loss 0.1664 | val Acc 98.04 F1 98.04 AUC 99.82 | 75s
round  9 | loss 0.1147 | val Acc 98.48 F1 98.48 AUC 99.87 | 75s
round 10 | loss 0.0910 | val Acc 98.60 F1 98.60 AUC 99.90 | 76s
round 11 | loss 0.0774 | val Acc 98.76 F1 98.76 AUC 99.92 | 75s
round 12 | loss 0.0676 | val Acc 98.84 F1 98.84 AUC 99.93 | 75s
round 13 | loss 0.0591 | val Acc 99.00 F1 99.00 AUC 99.94 | 75s
round 14

## 5. Sensitivity to the client partition

In [ ]:
# Ô 23 — FedAvg-LoRA với CÁCH CHIA CLIENT khác (kiểm tra phát hiện không phụ thuộc một cách chia)
import copy, time, os, torch, numpy as np
if "m" not in globals(): m = build()

ALPHA, SPLIT_SEED, SEED_RUN = 0.5, 1, 0     # chạy (0.5, 1, 0) trước; còn GPU thì (0.1, 1, 0)
K, ROUNDS, LOCAL_EP, MU = 5, 20, 1, 0.0
PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, SPLIT_SEED)
for k, p in enumerate(PARTS): print(f"client {k}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")
TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU}, split={SPLIT_SEED}, seed={SEED_RUN})"
CK = f"{FED}/ck_a{ALPHA}_mu{MU}_split{SPLIT_SEED}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = copy.deepcopy(INIT); best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m, g)
        losses.append(train_epochs(m, enc["train"], PARTS[k], epochs=LOCAL_EP, ref=g, mu=MU))
        st = get_state(m, KEYS)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m, g); v = evaluate(m, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), bad=0, best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m, best_g); res = eval_all(m, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG,
           parts=[[len(p), float(enc["train"].labels[p].mean())] for p in PARTS])
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{TAG.replace(' ','_')}_best.pt")

client 0: n= 1510 | fake= 22.2%
client 1: n= 3361 | fake=100.0%
client 2: n= 3137 | fake= 13.2%
client 3: n= 1158 | fake= 25.0%
client 4: n=  834 | fake= 72.2%


In [ ]:
# Ô 24.a — FedAvg-LoRA với CÁCH CHIA CLIENT khác (kiểm tra phát hiện không phụ thuộc một cách chia)
import copy, time, os, torch, numpy as np
if "m" not in globals(): m = build()

ALPHA, SPLIT_SEED, SEED_RUN = 0.5, 2, 0
K, ROUNDS, LOCAL_EP, MU = 5, 20, 1, 0.0
PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, SPLIT_SEED)
for k, p in enumerate(PARTS): print(f"client {k}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")
TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU}, split={SPLIT_SEED}, seed={SEED_RUN})"
CK = f"{FED}/ck_a{ALPHA}_mu{MU}_split{SPLIT_SEED}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = copy.deepcopy(INIT); best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m, g)
        losses.append(train_epochs(m, enc["train"], PARTS[k], epochs=LOCAL_EP, ref=g, mu=MU))
        st = get_state(m, KEYS)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m, g); v = evaluate(m, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), bad=0, best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m, best_g); res = eval_all(m, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG,
           parts=[[len(p), float(enc["train"].labels[p].mean())] for p in PARTS])
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{TAG.replace(' ','_')}_best.pt")

client 0: n= 1328 | fake= 27.3%
client 1: n=  893 | fake= 57.0%
client 2: n= 2200 | fake=  4.2%
client 3: n= 1709 | fake= 76.3%
client 4: n= 3870 | fake= 70.6%
round  1 | loss 0.7117 | val Acc 50.04 F1 66.68 AUC 88.25 | 85s
round  2 | loss 0.6994 | val Acc 50.00 F1 66.67 AUC 96.91 | 73s
round  3 | loss 0.6717 | val Acc 50.40 F1 66.84 AUC 98.65 | 74s
round  4 | loss 0.6135 | val Acc 77.28 F1 81.46 AUC 99.20 | 73s
round  5 | loss 0.5083 | val Acc 95.44 F1 95.58 AUC 99.42 | 73s
round  6 | loss 0.3592 | val Acc 97.52 F1 97.53 AUC 99.57 | 73s
round  7 | loss 0.2137 | val Acc 97.84 F1 97.84 AUC 99.66 | 73s
round  8 | loss 0.1250 | val Acc 98.24 F1 98.23 AUC 99.73 | 73s
round  9 | loss 0.0886 | val Acc 98.44 F1 98.43 AUC 99.78 | 72s
round 10 | loss 0.0725 | val Acc 98.48 F1 98.48 AUC 99.81 | 73s
round 11 | loss 0.0630 | val Acc 98.72 F1 98.71 AUC 99.84 | 73s
round 12 | loss 0.0554 | val Acc 98.76 F1 98.76 AUC 99.86 | 73s
round 13 | loss 0.0491 | val Acc 98.92 F1 98.92 AUC 99.87 | 74s
round 14

## 6. Centralized ablations: image augmentation, single modality, late fusion

In [ ]:
# Ô 24.b — Đợt 2: hàm dùng chung (chạy 1 lần)
import gc, time, torch, numpy as np, torch.nn.functional as F
import torchvision.transforms.v2.functional as TF

def augment(x):                       # x: (B, 3, 224, 224), đã chuẩn hóa theo CLIP
    out = []
    for xi in x:
        if torch.rand(1).item() < 0.5:                       # cắt ngẫu nhiên rồi phóng lại 224
            s = 0.6 + 0.4 * torch.rand(1).item(); c = int(224 * s ** 0.5)
            t, l = torch.randint(0, 224 - c + 1, (2,)).tolist()
            xi = TF.resized_crop(xi, t, l, c, c, [224, 224], antialias=True)
        if torch.rand(1).item() < 0.5:                       # thu nhỏ rồi phóng lại (mất chi tiết)
            r = int(torch.randint(96, 193, (1,)).item())
            xi = TF.resize(TF.resize(xi, [r, r], antialias=True), [224, 224], antialias=True)
        if torch.rand(1).item() < 0.3:                       # làm mờ
            xi = TF.gaussian_blur(xi, kernel_size=[5, 5], sigma=[0.1 + 1.4 * torch.rand(1).item()])
        if torch.rand(1).item() < 0.3:                       # nhiễu nhẹ
            xi = xi + 0.05 * torch.randn_like(xi)
        out.append(xi)
    return torch.stack(out)

def train_epochs_aug(m, dset, idx=None, epochs=1, lr=1e-5, wd=0.01, bs=16, ref=None, mu=0.0):
    assert mu == 0, "hàm này không có FedProx"
    m.train(); ps = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(ps, lr=lr, weight_decay=wd); sc = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    tot, n = 0.0, 0
    for _ in range(epochs):
        for b in loader(dset, bs, True, idx):
            b = batch_to(b); b["pixel_values"] = augment(b["pixel_values"])
            with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
                loss, _ = m(b["input_ids"], b["attention_mask"], b["pixel_values"], b["labels"])
            opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.step(opt); sc.update()
            tot += loss.item() * len(b["labels"]); n += len(b["labels"])
    return tot / max(n, 1)

class CLIPModalMFND(CLIPForMFND):     # chỉ dùng một nhánh: "image" hoặc "text"
    def __init__(self, mode):
        super().__init__(); self.mode = mode
    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        o = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, return_dict=True)
        h = o.image_embeds if self.mode == "image" else o.text_embeds
        logits = self.classifier(h)
        return (F.cross_entropy(logits, labels) if labels is not None else None), logits

def run_centralized(model, tag, train_fn, seed, save_name, epochs=5, patience=2):
    set_state(model, INIT); torch.manual_seed(1000 * seed)
    keys = [n for n, p in model.named_parameters() if p.requires_grad]
    best, bad, best_state, t0 = -1.0, 0, None, time.time()
    for ep in range(1, epochs + 1):
        l = train_fn(model, enc["train"], epochs=1); v = evaluate(model, enc["validation"])
        print(f"epoch {ep} loss {l:.4f} | val F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
        if v["f1"] > best: best, bad, best_state = v["f1"], 0, get_state(model, keys)
        else:
            bad += 1
            if bad >= patience: break
    set_state(model, best_state); res = eval_all(model, tag)
    res.update(time_s=time.time() - t0, epochs_run=ep)
    save_result(tag, res); torch.save(best_state, f"{FED}/{save_name}.pt"); return res
print("Đã định nghĩa xong hàm đợt 2")

Đã định nghĩa xong hàm đợt 2


In [ ]:
# Ô 25 — Tập trung + tăng cường ảnh
SEED_RUN = 1                          # chạy 1, rồi đổi thành 2
run_centralized(m, f"Centralized + tăng cường ảnh (seed={SEED_RUN})", train_epochs_aug, SEED_RUN, f"cen_aug_s{SEED_RUN}_best")

epoch 1 loss 0.6948 | val F1 89.77 AUC 96.63 | 90s
epoch 2 loss 0.3191 | val F1 96.05 AUC 99.44 | 162s
epoch 3 loss 0.1811 | val F1 97.85 AUC 99.77 | 232s
epoch 4 loss 0.1434 | val F1 98.13 AUC 99.85 | 303s
epoch 5 loss 0.1285 | val F1 98.33 AUC 99.90 | 374s
[Centralized + tăng cường ảnh (seed=1)] val F1 98.33 | test1 Acc/F1 99.6/99.6 | OOD Acc/F1/AUC 63.2/54.4/71.9


{'val': {'acc': 98.32,
  'f1': 98.32535885167464,
  'auc': np.float64(99.90275199999999)},
 'test1_nyt_mj': {'acc': 99.6, 'f1': 99.6, 'auc': np.float64(99.9952)},
 'test2_bbc_dalle': {'acc': 47.4,
  'f1': 29.86666666666667,
  'auc': np.float64(53.028000000000006)},
 'test3_cnn_dalle': {'acc': 54.6,
  'f1': 37.46556473829201,
  'auc': np.float64(65.8056)},
 'test4_bbc_sdxl': {'acc': 70.39999999999999,
  'f1': 70.39999999999999,
  'auc': np.float64(80.67360000000001)},
 'test5_cnn_sdxl': {'acc': 80.4,
  'f1': 79.83539094650206,
  'auc': np.float64(88.1736)},
 'OOD': {'acc': 63.199999999999996, 'f1': 54.39190558786518, 'auc': 71.9202},
 'time_s': 384.03704714775085,
 'epochs_run': 5}

In [ ]:
# Ô 26 — Tập trung, chỉ một nhánh
MODE, SEED_RUN = "text", 2      # chạy "image", rồi đổi thành "text"
if "m2" in globals(): del m2
gc.collect(); torch.cuda.empty_cache()
m2 = CLIPModalMFND(MODE).to(device)
run_centralized(m2, f"Centralized chỉ {'ảnh' if MODE == 'image' else 'chữ'} (seed={SEED_RUN})", train_epochs, SEED_RUN, f"cen_{MODE}_s{SEED_RUN}_best")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

epoch 1 loss 0.6936 | val F1 8.36 AUC 61.35 | 57s
epoch 2 loss 0.6871 | val F1 61.35 AUC 82.93 | 112s
epoch 3 loss 0.6421 | val F1 80.38 AUC 88.55 | 167s
epoch 4 loss 0.5359 | val F1 82.77 AUC 90.44 | 222s
epoch 5 loss 0.4768 | val F1 83.79 AUC 91.48 | 275s
[Centralized chỉ chữ (seed=2)] val F1 83.79 | test1 Acc/F1 88.6/88.4 | OOD Acc/F1/AUC 66.5/74.7/86.4


{'val': {'acc': 84.76, 'f1': 83.79413015737984, 'auc': np.float64(91.479552)},
 'test1_nyt_mj': {'acc': 88.6,
  'f1': 88.43813387423936,
  'auc': np.float64(95.1)},
 'test2_bbc_dalle': {'acc': 58.4,
  'f1': 70.45454545454545,
  'auc': np.float64(85.83840000000001)},
 'test3_cnn_dalle': {'acc': 74.0,
  'f1': 78.54785478547855,
  'auc': np.float64(86.47279999999999)},
 'test4_bbc_sdxl': {'acc': 58.4,
  'f1': 70.45454545454545,
  'auc': np.float64(86.1048)},
 'test5_cnn_sdxl': {'acc': 75.0,
  'f1': 79.27031509121062,
  'auc': np.float64(86.98559999999999)},
 'OOD': {'acc': 66.45, 'f1': 74.68181519644503, 'auc': 86.3504},
 'time_s': 285.39050483703613,
 'epochs_run': 5}

In [ ]:
# Ô 27 — Kết hợp muộn: trung bình xác suất của mô hình chỉ ảnh và chỉ chữ (không huấn luyện)
import gc, numpy as np, torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
SEED_RUN = 1

@torch.no_grad()
def probs(mm, dset, bs=64):
    mm.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            _, lg = mm(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

SPL = ["validation"] + TESTS; PR = {}
for mode in ["image", "text"]:
    if "m2" in globals(): del m2
    gc.collect(); torch.cuda.empty_cache()
    m2 = CLIPModalMFND(mode).to(device)
    set_state(m2, torch.load(f"{FED}/cen_{mode}_s{SEED_RUN}_best.pt"))
    PR[mode] = {s: probs(m2, enc[s]) for s in SPL}

res = {}
for s in SPL:
    y = PR["image"][s][1]; assert (y == PR["text"][s][1]).all()
    p = (PR["image"][s][0] + PR["text"][s][0]) / 2; pred = (p > 0.5).astype(int)
    res["val" if s == "validation" else s] = dict(acc=float(accuracy_score(y, pred) * 100),
                                                   f1=float(f1_score(y, pred) * 100), auc=float(roc_auc_score(y, p) * 100))
res["OOD"] = {k: float(np.mean([res[s][k] for s in TESTS[1:]])) for k in ("acc", "f1", "auc")}
TAG = f"Kết hợp muộn: TB xác suất ảnh + chữ (seed={SEED_RUN})"
print(f"[{TAG}] val F1 {res['val']['f1']:.2f} | test1 Acc/F1 {res[TESTS[0]]['acc']:.1f}/{res[TESTS[0]]['f1']:.1f} | "
      f"OOD Acc/F1/AUC {res['OOD']['acc']:.1f}/{res['OOD']['f1']:.1f}/{res['OOD']['auc']:.1f}")
save_result(TAG, res)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[Kết hợp muộn: TB xác suất ảnh + chữ (seed=1)] val F1 99.60 | test1 Acc/F1 99.8/99.8 | OOD Acc/F1/AUC 82.7/82.7/91.5


{'Local-only (a=0.5)': {'val': {'f1': 64.50185510719089},
  'test1_nyt_mj': {'acc': 69.12,
   'f1': 65.80695895503334,
   'auc': 89.00768000000001},
  'test2_bbc_dalle': {'acc': 55.879999999999995,
   'f1': 55.36127936914015,
   'auc': 64.49216},
  'test3_cnn_dalle': {'acc': 58.28000000000001,
   'f1': 57.070760573017274,
   'auc': 64.35679999999999},
  'test4_bbc_sdxl': {'acc': 56.239999999999995,
   'f1': 55.942704613903814,
   'auc': 67.21392},
  'test5_cnn_sdxl': {'acc': 59.2, 'f1': 57.79609742343625, 'auc': 69.54816},
  'OOD': {'acc': 57.4, 'f1': 56.542710494874385, 'auc': 66.40276},
  'clients': [{'val': {'acc': 50.0, 'f1': 0.0, 'auc': 83.31728000000001},
    'test1_nyt_mj': {'acc': 50.0, 'f1': 0.0, 'auc': 87.8},
    'test2_bbc_dalle': {'acc': 50.0, 'f1': 0.0, 'auc': 57.36559999999999},
    'test3_cnn_dalle': {'acc': 50.0, 'f1': 0.0, 'auc': 75.2928},
    'test4_bbc_sdxl': {'acc': 50.0, 'f1': 0.0, 'auc': 45.004},
    'test5_cnn_sdxl': {'acc': 50.0, 'f1': 0.0, 'auc': 64.7088},
    

In [ ]:
# Ô 29 — Kết hợp muộn: trung bình xác suất của mô hình chỉ ảnh và chỉ chữ (không huấn luyện)
import gc, numpy as np, torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

@torch.no_grad()
def probs(mm, dset, bs=64):
    mm.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            _, lg = mm(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def late_fusion(path_img, path_txt, TAG):
    SPL = ["validation"] + TESTS; PR = {}
    for mode, path in [("image", path_img), ("text", path_txt)]:
        mm = CLIPModalMFND(mode).to(device); set_state(mm, INIT); set_state(mm, torch.load(path))
        PR[mode] = {s: probs(mm, enc[s]) for s in SPL}
        del mm; gc.collect(); torch.cuda.empty_cache()
    res = {}
    for s in SPL:
        y = PR["image"][s][1]; assert (y == PR["text"][s][1]).all()
        p = (PR["image"][s][0] + PR["text"][s][0]) / 2; pred = (p > 0.5).astype(int)
        res["val" if s == "validation" else s] = dict(acc=float(accuracy_score(y, pred) * 100),
                                                       f1=float(f1_score(y, pred) * 100), auc=float(roc_auc_score(y, p) * 100))
    res["OOD"] = {k: float(np.mean([res[s][k] for s in TESTS[1:]])) for k in ("acc", "f1", "auc")}
    print(f"[{TAG}] val F1 {res['val']['f1']:.2f} | test1 Acc/F1 {res[TESTS[0]]['acc']:.1f}/{res[TESTS[0]]['f1']:.1f} | "
          f"OOD Acc/F1/AUC {res['OOD']['acc']:.1f}/{res['OOD']['f1']:.1f}/{res['OOD']['auc']:.1f}")
    save_result(TAG, res)

# A — # Tập trung, seed 2 (seed 1 đã chạy ở ô 27)
late_fusion(f"{FED}/cen_image_s2_best.pt", f"{FED}/cen_text_s2_best.pt",
            "Kết hợp muộn: TB xác suất ảnh + chữ (seed=2)")

# B — # Kết hợp muộn trong học liên kết: xem ô 29b (split 0) và ô 29c (split 2)
# late_fusion(f"{FED}/fed_image_a0.5_split0_s0_best.pt", f"{FED}/fed_text_a0.5_split0_s0_best.pt",
#             "FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=0, seed=0)")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[Kết hợp muộn: TB xác suất ảnh + chữ (seed=2)] val F1 99.60 | test1 Acc/F1 99.8/99.8 | OOD Acc/F1/AUC 81.5/80.7/90.9


## 7. Late fusion under federated learning (FedSVLM-LF)
Image-only and text-only models trained with FedAvg; their probabilities are averaged.

In [ ]:
# Ô 28a — FedAvg-LoRA cho mô hình một nhánh (ảnh hoặc chữ)
import copy, time, os, gc, torch, numpy as np
MODE, ALPHA, SPLIT_SEED, SEED_RUN = "image", 0.5, 0, 0
K, ROUNDS, LOCAL_EP = 5, 20, 1
if "m2" in globals(): del m2
gc.collect(); torch.cuda.empty_cache()
m2 = CLIPModalMFND(MODE).to(device); set_state(m2, INIT)

part = "vision_model" if MODE == "image" else "text_model"   # chỉ gửi LoRA của nhánh đang dùng + lớp phân loại
KEYS_M = [k for k in KEYS if ("lora_" in k and part in k) or k.startswith("classifier")]
n_upd = sum(INIT[k].numel() for k in KEYS_M)
print(f"{MODE}: gửi {n_upd:,} tham số ≈ {n_upd*4/1e6:.2f} MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)")

PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, SPLIT_SEED)
TAG = f"FedAvg-LoRA chỉ {'ảnh' if MODE == 'image' else 'chữ'} (a={ALPHA}, split={SPLIT_SEED}, seed={SEED_RUN})"
CK = f"{FED}/ck_fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = {k: INIT[k].clone() for k in KEYS_M}; best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m2, g)
        losses.append(train_epochs(m2, enc["train"], PARTS[k], epochs=LOCAL_EP))
        st = get_state(m2, KEYS_M)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m2, g); v = evaluate(m2, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m2, best_g); res = eval_all(m2, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), update_mb=n_upd * 4 / 1e6, log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}_best.pt")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

image: gửi 295,938 tham số ≈ 1.18 MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)
round  1 | loss 0.6929 | val Acc 51.16 F1 8.40 AUC 56.36 | 67s
round  2 | loss 0.6848 | val Acc 51.56 F1 7.77 AUC 70.93 | 57s
round  3 | loss 0.6644 | val Acc 53.44 F1 13.65 AUC 88.01 | 57s
round  4 | loss 0.6290 | val Acc 57.16 F1 25.37 AUC 95.67 | 57s
round  5 | loss 0.5866 | val Acc 55.88 F1 21.38 AUC 96.33 | 57s
round  6 | loss 0.5515 | val Acc 59.44 F1 32.04 AUC 97.04 | 57s
round  7 | loss 0.5171 | val Acc 57.20 F1 25.38 AUC 97.48 | 57s
round  8 | loss 0.4891 | val Acc 56.60 F1 23.32 AUC 98.14 | 56s
round  9 | loss 0.4650 | val Acc 53.96 F1 14.68 AUC 98.49 | 56s
round 10 | loss 0.4458 | val Acc 55.88 F1 21.05 AUC 98.99 | 58s
round 11 | loss 0.4330 | val Acc 56.24 F1 22.19 AUC 99.21 | 56s
round 12 | loss 0.4099 | val Acc 60.24 F1 34.08 AUC 99.38 | 56s
round 13 | loss 0.3893 | val Acc 58.64 F1 29.47 AUC 99.44 | 56s
round 14 | loss 0.3782 | val Acc 65.76 F1 48.00 AUC 99.55 | 53s
round 15 | loss 0.3488 | val Ac

In [ ]:
# Ô 28b — FedAvg-LoRA cho mô hình một nhánh: CHỮ (a=0.5, split 0, seed 0)
import copy, time, os, gc, torch, numpy as np
MODE, ALPHA, SPLIT_SEED, SEED_RUN = "text", 0.5, 0, 0
K, ROUNDS, LOCAL_EP = 5, 20, 1
if "m2" in globals(): del m2
gc.collect(); torch.cuda.empty_cache()
m2 = CLIPModalMFND(MODE).to(device); set_state(m2, INIT)

part = "vision_model" if MODE == "image" else "text_model"   # chỉ gửi LoRA của nhánh đang dùng + lớp phân loại
KEYS_M = [k for k in KEYS if ("lora_" in k and part in k) or k.startswith("classifier")]
n_upd = sum(INIT[k].numel() for k in KEYS_M)
print(f"{MODE}: gửi {n_upd:,} tham số ≈ {n_upd*4/1e6:.2f} MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)")

PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, SPLIT_SEED)
TAG = f"FedAvg-LoRA chỉ {'ảnh' if MODE == 'image' else 'chữ'} (a={ALPHA}, split={SPLIT_SEED}, seed={SEED_RUN})"
CK = f"{FED}/ck_fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = {k: INIT[k].clone() for k in KEYS_M}; best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m2, g)
        losses.append(train_epochs(m2, enc["train"], PARTS[k], epochs=LOCAL_EP))
        st = get_state(m2, KEYS_M)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m2, g); v = evaluate(m2, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m2, best_g); res = eval_all(m2, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), update_mb=n_upd * 4 / 1e6, log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}_best.pt")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

text: gửi 197,634 tham số ≈ 0.79 MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)
round  1 | loss 0.6959 | val Acc 49.04 F1 2.75 AUC 47.84 | 72s
round  2 | loss 0.6934 | val Acc 49.76 F1 2.18 AUC 50.99 | 79s
round  3 | loss 0.6901 | val Acc 49.80 F1 1.26 AUC 54.78 | 57s
round  4 | loss 0.6849 | val Acc 49.80 F1 1.10 AUC 59.40 | 55s
round  5 | loss 0.6756 | val Acc 50.48 F1 3.73 AUC 65.28 | 54s
round  6 | loss 0.6607 | val Acc 54.80 F1 19.40 AUC 69.44 | 53s
round  7 | loss 0.6389 | val Acc 60.64 F1 40.00 AUC 70.83 | 52s
round  8 | loss 0.6144 | val Acc 65.12 F1 58.04 AUC 71.78 | 52s
round  9 | loss 0.5887 | val Acc 65.28 F1 65.11 AUC 71.55 | 53s
round 10 | loss 0.5640 | val Acc 63.72 F1 68.99 AUC 71.60 | 53s
round 11 | loss 0.5434 | val Acc 64.56 F1 69.13 AUC 71.79 | 52s
round 12 | loss 0.5244 | val Acc 64.08 F1 69.64 AUC 72.08 | 52s
round 13 | loss 0.5071 | val Acc 63.68 F1 70.05 AUC 72.28 | 52s
round 14 | loss 0.4941 | val Acc 65.48 F1 69.88 AUC 72.38 | 53s
round 15 | loss 0.4817 | val Acc 66

In [ ]:
# Ô 28c — FedAvg-LoRA cho mô hình một nhánh: ẢNH (a=0.5, split 2, seed 0)
import copy, time, os, gc, torch, numpy as np
MODE, ALPHA, SPLIT_SEED, SEED_RUN = "image", 0.5, 2, 0
K, ROUNDS, LOCAL_EP = 5, 20, 1
if "m2" in globals(): del m2
gc.collect(); torch.cuda.empty_cache()
m2 = CLIPModalMFND(MODE).to(device); set_state(m2, INIT)

part = "vision_model" if MODE == "image" else "text_model"   # chỉ gửi LoRA của nhánh đang dùng + lớp phân loại
KEYS_M = [k for k in KEYS if ("lora_" in k and part in k) or k.startswith("classifier")]
n_upd = sum(INIT[k].numel() for k in KEYS_M)
print(f"{MODE}: gửi {n_upd:,} tham số ≈ {n_upd*4/1e6:.2f} MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)")

PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, SPLIT_SEED)
for k, p in enumerate(PARTS): print(f"client {k}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")
TAG = f"FedAvg-LoRA chỉ {'ảnh' if MODE == 'image' else 'chữ'} (a={ALPHA}, split={SPLIT_SEED}, seed={SEED_RUN})"
CK = f"{FED}/ck_fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = {k: INIT[k].clone() for k in KEYS_M}; best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m2, g)
        losses.append(train_epochs(m2, enc["train"], PARTS[k], epochs=LOCAL_EP))
        st = get_state(m2, KEYS_M)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m2, g); v = evaluate(m2, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m2, best_g); res = eval_all(m2, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), update_mb=n_upd * 4 / 1e6, log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}_best.pt")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

image: gửi 295,938 tham số ≈ 1.18 MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)
client 0: n= 1328 | fake= 27.3%
client 1: n=  893 | fake= 57.0%
client 2: n= 2200 | fake=  4.2%
client 3: n= 1709 | fake= 76.3%
client 4: n= 3870 | fake= 70.6%
round  1 | loss 0.6875 | val Acc 52.80 F1 23.58 AUC 57.61 | 67s
round  2 | loss 0.6835 | val Acc 67.28 F1 65.60 AUC 74.15 | 56s
round  3 | loss 0.6756 | val Acc 67.24 F1 75.02 AUC 90.83 | 55s
round  4 | loss 0.6594 | val Acc 63.96 F1 73.49 AUC 97.25 | 53s
round  5 | loss 0.6327 | val Acc 68.80 F1 76.21 AUC 98.96 | 53s
round  6 | loss 0.6005 | val Acc 80.00 F1 83.31 AUC 99.43 | 53s
round  7 | loss 0.5703 | val Acc 89.72 F1 90.67 AUC 99.62 | 53s
round  8 | loss 0.5412 | val Acc 94.48 F1 94.76 AUC 99.73 | 52s
round  9 | loss 0.5102 | val Acc 96.80 F1 96.88 AUC 99.77 | 51s
round 10 | loss 0.4751 | val Acc 97.96 F1 97.99 AUC 99.80 | 52s
round 11 | loss 0.4350 | val Acc 98.32 F1 98.34 AUC 99.82 | 53s
round 12 | loss 0.3904 | val Acc 98.72 F1 98.73 AUC 99.84 | 5

In [ ]:
# Ô 28d — FedAvg-LoRA cho mô hình một nhánh: CHỮ (a=0.5, split 2, seed 0)
import copy, time, os, gc, torch, numpy as np
MODE, ALPHA, SPLIT_SEED, SEED_RUN = "text", 0.5, 2, 0
K, ROUNDS, LOCAL_EP = 5, 20, 1
if "m2" in globals(): del m2
gc.collect(); torch.cuda.empty_cache()
m2 = CLIPModalMFND(MODE).to(device); set_state(m2, INIT)

part = "vision_model" if MODE == "image" else "text_model"   # chỉ gửi LoRA của nhánh đang dùng + lớp phân loại
KEYS_M = [k for k in KEYS if ("lora_" in k and part in k) or k.startswith("classifier")]
n_upd = sum(INIT[k].numel() for k in KEYS_M)
print(f"{MODE}: gửi {n_upd:,} tham số ≈ {n_upd*4/1e6:.2f} MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)")

PARTS = dirichlet_split(enc["train"].labels, K, ALPHA, SPLIT_SEED)
for k, p in enumerate(PARTS): print(f"client {k}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")
TAG = f"FedAvg-LoRA chỉ {'ảnh' if MODE == 'image' else 'chữ'} (a={ALPHA}, split={SPLIT_SEED}, seed={SEED_RUN})"
CK = f"{FED}/ck_fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False)
    g, best, best_g, LOG, r0 = S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    print(f"Tiếp tục từ vòng {r0+1}")
else:
    g = {k: INIT[k].clone() for k in KEYS_M}; best, best_g, LOG, r0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + r0)
N = sum(len(p) for p in PARTS)
for r in range(r0 + 1, ROUNDS + 1):
    t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
    for k in range(K):
        set_state(m2, g)
        losses.append(train_epochs(m2, enc["train"], PARTS[k], epochs=LOCAL_EP))
        st = get_state(m2, KEYS_M)
        for key in acc: acc[key] += st[key] * (len(PARTS[k]) / N)
    g = acc; set_state(m2, g); v = evaluate(m2, enc["validation"])
    LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                    loss=float(np.mean(losses)), sec=float(time.time() - t0)))
    print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
    torch.save(dict(g=g, best=float(best), best_g=best_g, LOG=LOG, r=int(r)), CK)
set_state(m2, best_g); res = eval_all(m2, TAG)
res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), update_mb=n_upd * 4 / 1e6, log=LOG)
save_result(TAG, res); torch.save(best_g, f"{FED}/fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}_best.pt")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

text: gửi 197,634 tham số ≈ 0.79 MB mỗi lượt (Light-MFND đầy đủ: 8.27 MB)
client 0: n= 1328 | fake= 27.3%
client 1: n=  893 | fake= 57.0%
client 2: n= 2200 | fake=  4.2%
client 3: n= 1709 | fake= 76.3%
client 4: n= 3870 | fake= 70.6%
round  1 | loss 0.6894 | val Acc 48.12 F1 10.98 AUC 47.03 | 72s
round  2 | loss 0.6884 | val Acc 48.20 F1 29.43 AUC 49.08 | 64s
round  3 | loss 0.6874 | val Acc 51.48 F1 54.28 AUC 51.68 | 57s
round  4 | loss 0.6861 | val Acc 51.76 F1 64.76 AUC 55.20 | 56s
round  5 | loss 0.6842 | val Acc 50.72 F1 66.76 AUC 60.17 | 55s
round  6 | loss 0.6812 | val Acc 50.16 F1 66.72 AUC 65.73 | 55s
round  7 | loss 0.6778 | val Acc 50.08 F1 66.70 AUC 70.44 | 55s
round  8 | loss 0.6748 | val Acc 50.00 F1 66.67 AUC 74.22 | 56s
round  9 | loss 0.6715 | val Acc 50.00 F1 66.67 AUC 77.92 | 58s
round 10 | loss 0.6661 | val Acc 50.00 F1 66.67 AUC 81.14 | 54s
round 11 | loss 0.6575 | val Acc 50.04 F1 66.68 AUC 83.59 | 55s
round 12 | loss 0.6465 | val Acc 50.08 F1 66.70 AUC 85.27 | 57

In [ ]:
# Ô 29b — Kết hợp muộn trong học liên kết (FedAvg α=0.5, split 0, seed 0)
late_fusion(f"{FED}/fed_image_a0.5_split0_s0_best.pt", f"{FED}/fed_text_a0.5_split0_s0_best.pt",
            "FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=0, seed=0)")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=0, seed=0)] val F1 97.06 | test1 Acc/F1 98.8/98.8 | OOD Acc/F1/AUC 71.2/64.9/86.9


In [ ]:
# Ô 29c — Kết hợp muộn trong học liên kết (FedAvg α=0.5, split 2, seed 0)
late_fusion(f"{FED}/fed_image_a0.5_split2_s0_best.pt", f"{FED}/fed_text_a0.5_split2_s0_best.pt",
            "FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=2, seed=0)")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=2, seed=0)] val F1 99.28 | test1 Acc/F1 99.8/99.8 | OOD Acc/F1/AUC 82.0/82.2/92.0


## 8. Extended experiments
Shared helpers, TF-IDF + SVM baseline, 40 rounds, more seeds for split 2, FedProx, an ensemble control, and a check for duplicated data across the OOD test sets.

*Cell 41a was written to prepare an extra client built from test2/test3; that experiment was dropped because the check found that test2/test4 and test3/test5 share most captions and real images.*

In [ ]:
# Ô 32 — Hàm dùng chung cho các thí nghiệm mở rộng (chạy 1 lần)
import copy, time, os, gc, shutil, torch, numpy as np

def _load_ck(CK, init_g):
    if os.path.exists(CK):
        S = torch.load(CK, weights_only=False); print(f"Tiếp tục từ vòng {S['r']+1}")
        return S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    return init_g, -1.0, None, [], 0

def _fedavg_loop(model, keys, parts, g, best, best_g, LOG, r0, ROUNDS, seed, CK, mu=0.0):
    torch.manual_seed(1000 * seed + r0); N = sum(len(p) for p in parts)
    for r in range(r0 + 1, ROUNDS + 1):
        t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
        for p in parts:
            set_state(model, g)
            losses.append(train_epochs(model, enc["train"], p, epochs=1, ref=g, mu=mu))
            st = get_state(model, keys)
            for key in acc: acc[key] += st[key] * (len(p) / N)
        g = acc; set_state(model, g); v = evaluate(model, enc["validation"])
        LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                        loss=float(np.mean(losses)), sec=float(time.time() - t0)))
        print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
        if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
        torch.save(dict(g=g, best=float(best), best_g=best_g, LOG=LOG, r=int(r)), CK)
    return best_g, LOG

def _finish(model, best_g, LOG, TAG, mb, path):
    set_state(model, best_g); res = eval_all(model, TAG)
    res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
               time_s=float(sum(x["sec"] for x in LOG)), update_mb=mb, log=LOG)
    save_result(TAG, res); torch.save(best_g, path); return path

def run_fed_light(ALPHA, SPLIT_SEED=0, SEED_RUN=0, MU=0.0, K=5, ROUNDS=20):
    """FedAvg/FedProx với Light-MFND đầy đủ."""
    global m
    if "m" not in globals(): m = build()
    parts = dirichlet_split(enc["train"].labels, K, ALPHA, SPLIT_SEED)
    for i, p in enumerate(parts): print(f"client {i}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")
    TAG = f"FedAvg-LoRA (a={ALPHA}, mu={MU}, split={SPLIT_SEED}, seed={SEED_RUN}, K={K})"
    CK = f"{FED}/ck_light_a{ALPHA}_mu{MU}_split{SPLIT_SEED}_s{SEED_RUN}_K{K}.pt"
    g, best, best_g, LOG, r0 = _load_ck(CK, {k: INIT[k].clone() for k in KEYS})
    best_g, LOG = _fedavg_loop(m, KEYS, parts, g, best, best_g, LOG, r0, ROUNDS, SEED_RUN, CK, mu=MU)
    return _finish(m, best_g, LOG, TAG, 8.27, f"{FED}/light_a{ALPHA}_mu{MU}_split{SPLIT_SEED}_s{SEED_RUN}_K{K}_best.pt")

def run_fed_modal(MODE, ALPHA=0.5, SPLIT_SEED=0, SEED_RUN=0, ROUNDS=20, continue_from=None):
    """FedAvg cho mô hình một nhánh (ảnh hoặc chữ). continue_from: checkpoint cũ để chạy tiếp thay vì từ đầu."""
    global m2
    if "m2" in globals(): del m2
    gc.collect(); torch.cuda.empty_cache()
    m2 = CLIPModalMFND(MODE).to(device); set_state(m2, INIT)
    part = "vision_model" if MODE == "image" else "text_model"
    keys = [k for k in KEYS if ("lora_" in k and part in k) or k.startswith("classifier")]
    mb = sum(INIT[k].numel() for k in keys) * 4 / 1e6
    parts = dirichlet_split(enc["train"].labels, 5, ALPHA, SPLIT_SEED)
    for i, p in enumerate(parts): print(f"client {i}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")
    sfx, rtag = ("", "") if ROUNDS == 20 else (f", {ROUNDS} vòng", f"_r{ROUNDS}")
    TAG = f"FedAvg-LoRA chỉ {'ảnh' if MODE == 'image' else 'chữ'} (a={ALPHA}, split={SPLIT_SEED}, seed={SEED_RUN}{sfx})"
    CK = f"{FED}/ck_fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}{rtag}.pt"
    if continue_from and not os.path.exists(CK):
        shutil.copy(continue_from, CK); print("Chép checkpoint cũ để chạy tiếp:", continue_from)
    g, best, best_g, LOG, r0 = _load_ck(CK, {k: INIT[k].clone() for k in keys})
    best_g, LOG = _fedavg_loop(m2, keys, parts, g, best, best_g, LOG, r0, ROUNDS, SEED_RUN, CK)
    return _finish(m2, best_g, LOG, TAG, mb, f"{FED}/fed_{MODE}_a{ALPHA}_split{SPLIT_SEED}_s{SEED_RUN}{rtag}_best.pt")
print("Đã định nghĩa run_fed_light, run_fed_modal")

Đã định nghĩa run_fed_light, run_fed_modal


In [ ]:
# Ô 33 — Mốc SVM + TF-IDF tự chạy (chỉ dùng chữ; chú thích giải mã lại từ token CLIP)
import numpy as np
from transformers import CLIPTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
tok = CLIPTokenizer.from_pretrained(MODEL_NAME); SRC = f"{FED}/npy"
def load_text(sp):
    ids, msk, y = np.load(f"{SRC}/{sp}_ids.npy"), np.load(f"{SRC}/{sp}_msk.npy"), np.load(f"{SRC}/{sp}_lab.npy")
    txt = [tok.decode(i[mk.astype(bool)].tolist(), skip_special_tokens=True) for i, mk in zip(ids, msk)]
    return txt, y, float((msk.sum(1) == msk.shape[1]).mean() * 100)
D = {sp: load_text(sp) for sp in SPLITS}
for sp in SPLITS: print(f"{sp:18s} n={len(D[sp][1]):5d} | dài tối đa (bị cắt): {D[sp][2]:.1f}%")
print("Ví dụ chú thích:", D["train"][0][0][:150])
vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))          # cùng cấu hình repo Light-MFND
clf = LinearSVC(C=1.0, class_weight="balanced").fit(vec.fit_transform(D["train"][0]), D["train"][1])
res = {}
for sp in ["validation"] + TESTS:
    s = clf.decision_function(vec.transform(D[sp][0])); y = D[sp][1]; pred = (s > 0).astype(int)
    res["val" if sp == "validation" else sp] = dict(acc=float(accuracy_score(y, pred) * 100),
                                                     f1=float(f1_score(y, pred) * 100), auc=float(roc_auc_score(y, s) * 100))
res["OOD"] = {k: float(np.mean([res[s_][k] for s_ in TESTS[1:]])) for k in ("acc", "f1", "auc")}
TAG = "SVM + TF-IDF (tự chạy, chữ giải mã từ token CLIP)"
print(f"[{TAG}] val F1 {res['val']['f1']:.2f} | test1 Acc/F1 {res[TESTS[0]]['acc']:.1f}/{res[TESTS[0]]['f1']:.1f} | "
      f"OOD Acc/F1/AUC {res['OOD']['acc']:.1f}/{res['OOD']['f1']:.1f}/{res['OOD']['auc']:.1f}")
save_result(TAG, res)

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

train              n=10000 | dài tối đa (bị cắt): 0.3%
validation         n= 2500 | dài tối đa (bị cắt): 0.2%
test1_nyt_mj       n=  500 | dài tối đa (bị cắt): 0.2%
test2_bbc_dalle    n=  500 | dài tối đa (bị cắt): 0.0%
test3_cnn_dalle    n=  500 | dài tối đa (bị cắt): 0.0%
test4_bbc_sdxl     n=  500 | dài tối đa (bị cắt): 0.0%
test5_cnn_sdxl     n=  500 | dài tối đa (bị cắt): 0.0%
Ví dụ chú thích: andal ampatuan jr ., a notorious crime lord in the southern philippines , laughs off 4 1 murder charges at his trial 's opening on tuesday .
[SVM + TF-IDF (tự chạy, chữ giải mã từ token CLIP)] val F1 80.50 | test1 Acc/F1 86.2/86.4 | OOD Acc/F1/AUC 75.8/78.3/87.0


{'Local-only (a=0.5)': {'val': {'f1': 64.50185510719089},
  'test1_nyt_mj': {'acc': 69.12,
   'f1': 65.80695895503334,
   'auc': 89.00768000000001},
  'test2_bbc_dalle': {'acc': 55.879999999999995,
   'f1': 55.36127936914015,
   'auc': 64.49216},
  'test3_cnn_dalle': {'acc': 58.28000000000001,
   'f1': 57.070760573017274,
   'auc': 64.35679999999999},
  'test4_bbc_sdxl': {'acc': 56.239999999999995,
   'f1': 55.942704613903814,
   'auc': 67.21392},
  'test5_cnn_sdxl': {'acc': 59.2, 'f1': 57.79609742343625, 'auc': 69.54816},
  'OOD': {'acc': 57.4, 'f1': 56.542710494874385, 'auc': 66.40276},
  'clients': [{'val': {'acc': 50.0, 'f1': 0.0, 'auc': 83.31728000000001},
    'test1_nyt_mj': {'acc': 50.0, 'f1': 0.0, 'auc': 87.8},
    'test2_bbc_dalle': {'acc': 50.0, 'f1': 0.0, 'auc': 57.36559999999999},
    'test3_cnn_dalle': {'acc': 50.0, 'f1': 0.0, 'auc': 75.2928},
    'test4_bbc_sdxl': {'acc': 50.0, 'f1': 0.0, 'auc': 45.004},
    'test5_cnn_sdxl': {'acc': 50.0, 'f1': 0.0, 'auc': 64.7088},
    

In [ ]:
# Ô 34 — FedAvg nhánh ẢNH, split 0, chạy tiếp đến 40 vòng
run_fed_modal("image", SPLIT_SEED=0, ROUNDS=40, continue_from=f"{FED}/ck_fed_image_a0.5_split0_s0.pt")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

client 0: n= 3777 | fake=  0.8%
client 1: n= 1699 | fake= 64.0%
client 2: n= 3414 | fake= 83.6%
client 3: n=  290 | fake= 85.2%
client 4: n=  820 | fake= 95.2%
Chép checkpoint cũ để chạy tiếp: /content/drive/MyDrive/NLP_project/fed/ck_fed_image_a0.5_split0_s0.pt
Tiếp tục từ vòng 21
round 21 | loss 0.2094 | val Acc 97.56 F1 97.52 AUC 99.80 | 109s
round 22 | loss 0.1936 | val Acc 98.04 F1 98.03 AUC 99.83 | 118s
round 23 | loss 0.1793 | val Acc 97.92 F1 97.90 AUC 99.84 | 58s
round 24 | loss 0.1703 | val Acc 98.20 F1 98.19 AUC 99.86 | 57s
round 25 | loss 0.1595 | val Acc 98.44 F1 98.44 AUC 99.86 | 55s
round 26 | loss 0.1469 | val Acc 98.40 F1 98.39 AUC 99.88 | 55s
round 27 | loss 0.1377 | val Acc 98.36 F1 98.35 AUC 99.89 | 57s
round 28 | loss 0.1302 | val Acc 98.32 F1 98.31 AUC 99.90 | 59s
round 29 | loss 0.1218 | val Acc 98.52 F1 98.51 AUC 99.91 | 56s
round 30 | loss 0.1134 | val Acc 98.60 F1 98.59 AUC 99.91 | 55s
round 31 | loss 0.1097 | val Acc 98.56 F1 98.55 AUC 99.92 | 56s
round 32 | 

'/content/drive/MyDrive/NLP_project/fed/fed_image_a0.5_split0_s0_r40_best.pt'

In [ ]:
# Ô 35 — FedAvg nhánh CHỮ, split 0, chạy tiếp đến 40 vòng
run_fed_modal("text", SPLIT_SEED=0, ROUNDS=40, continue_from=f"{FED}/ck_fed_text_a0.5_split0_s0.pt")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

client 0: n= 3777 | fake=  0.8%
client 1: n= 1699 | fake= 64.0%
client 2: n= 3414 | fake= 83.6%
client 3: n=  290 | fake= 85.2%
client 4: n=  820 | fake= 95.2%
Chép checkpoint cũ để chạy tiếp: /content/drive/MyDrive/NLP_project/fed/ck_fed_text_a0.5_split0_s0.pt
Tiếp tục từ vòng 21
round 21 | loss 0.4460 | val Acc 66.76 F1 63.63 AUC 73.24 | 53s
round 22 | loss 0.4376 | val Acc 65.52 F1 57.45 AUC 73.31 | 53s
round 23 | loss 0.4362 | val Acc 64.96 F1 55.17 AUC 73.41 | 52s
round 24 | loss 0.4314 | val Acc 65.40 F1 56.64 AUC 73.66 | 53s
round 25 | loss 0.4303 | val Acc 64.48 F1 53.51 AUC 73.83 | 53s
round 26 | loss 0.4247 | val Acc 64.16 F1 52.54 AUC 74.10 | 53s
round 27 | loss 0.4216 | val Acc 65.52 F1 56.24 AUC 74.43 | 54s
round 28 | loss 0.4113 | val Acc 66.00 F1 56.94 AUC 74.77 | 54s
round 29 | loss 0.4079 | val Acc 66.44 F1 57.60 AUC 75.08 | 52s
round 30 | loss 0.4089 | val Acc 63.24 F1 47.46 AUC 75.25 | 53s
round 31 | loss 0.4260 | val Acc 63.60 F1 47.76 AUC 75.61 | 53s
round 32 | los

'/content/drive/MyDrive/NLP_project/fed/fed_text_a0.5_split0_s0_r40_best.pt'

In [ ]:
# Ô 36 — Kết hợp muộn, split 0, 40 vòng
late_fusion(f"{FED}/fed_image_a0.5_split0_s0_r40_best.pt", f"{FED}/fed_text_a0.5_split0_s0_r40_best.pt",
            "FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=0, seed=0, 40 vòng)")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=0, seed=0, 40 vòng)] val F1 98.99 | test1 Acc/F1 99.8/99.8 | OOD Acc/F1/AUC 75.0/71.5/87.9


In [ ]:
# Ô 37 — Kết hợp muộn FL, split 2, seed 1 và 2 (mỗi seed: ảnh + chữ + kết hợp)
for s in (1, 2):
    pi = run_fed_modal("image", SPLIT_SEED=2, SEED_RUN=s)
    pt = run_fed_modal("text", SPLIT_SEED=2, SEED_RUN=s)
    late_fusion(pi, pt, f"FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=2, seed={s})")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

client 0: n= 1328 | fake= 27.3%
client 1: n=  893 | fake= 57.0%
client 2: n= 2200 | fake=  4.2%
client 3: n= 1709 | fake= 76.3%
client 4: n= 3870 | fake= 70.6%
round  1 | loss 0.6874 | val Acc 52.84 F1 23.89 AUC 57.62 | 81s
round  2 | loss 0.6834 | val Acc 67.40 F1 66.03 AUC 74.17 | 56s
round  3 | loss 0.6757 | val Acc 66.68 F1 74.73 AUC 90.75 | 55s
round  4 | loss 0.6597 | val Acc 63.24 F1 73.11 AUC 97.21 | 57s
round  5 | loss 0.6324 | val Acc 67.40 F1 75.40 AUC 98.92 | 53s
round  6 | loss 0.6005 | val Acc 79.44 F1 82.92 AUC 99.39 | 53s
round  7 | loss 0.5719 | val Acc 88.40 F1 89.60 AUC 99.61 | 53s
round  8 | loss 0.5437 | val Acc 93.52 F1 93.91 AUC 99.71 | 52s
round  9 | loss 0.5140 | val Acc 96.56 F1 96.66 AUC 99.76 | 52s
round 10 | loss 0.4801 | val Acc 97.68 F1 97.72 AUC 99.79 | 53s
round 11 | loss 0.4439 | val Acc 98.28 F1 98.30 AUC 99.81 | 54s
round 12 | loss 0.4011 | val Acc 98.68 F1 98.69 AUC 99.83 | 54s
round 13 | loss 0.3576 | val Acc 98.72 F1 98.72 AUC 99.85 | 54s
round 14

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

client 0: n= 1328 | fake= 27.3%
client 1: n=  893 | fake= 57.0%
client 2: n= 2200 | fake=  4.2%
client 3: n= 1709 | fake= 76.3%
client 4: n= 3870 | fake= 70.6%
round  1 | loss 0.6893 | val Acc 48.20 F1 11.24 AUC 47.06 | 54s
round  2 | loss 0.6884 | val Acc 48.36 F1 29.88 AUC 49.16 | 53s
round  3 | loss 0.6874 | val Acc 51.72 F1 54.57 AUC 51.81 | 54s
round  4 | loss 0.6862 | val Acc 51.96 F1 64.77 AUC 55.41 | 54s
round  5 | loss 0.6838 | val Acc 50.68 F1 66.77 AUC 60.46 | 55s
round  6 | loss 0.6808 | val Acc 50.16 F1 66.72 AUC 66.12 | 56s
round  7 | loss 0.6777 | val Acc 50.08 F1 66.70 AUC 70.86 | 57s
round  8 | loss 0.6748 | val Acc 50.00 F1 66.67 AUC 74.89 | 58s
round  9 | loss 0.6702 | val Acc 50.00 F1 66.67 AUC 78.63 | 57s
round 10 | loss 0.6640 | val Acc 50.04 F1 66.68 AUC 81.74 | 56s
round 11 | loss 0.6546 | val Acc 50.04 F1 66.68 AUC 83.91 | 54s
round 12 | loss 0.6429 | val Acc 50.24 F1 66.77 AUC 85.35 | 53s
round 13 | loss 0.6296 | val Acc 53.16 F1 68.01 AUC 86.50 | 54s
round 14

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

... (4 outputs left out)

client 0: n= 1328 | fake= 27.3%
client 1: n=  893 | fake= 57.0%
client 2: n= 2200 | fake=  4.2%
client 3: n= 1709 | fake= 76.3%
client 4: n= 3870 | fake= 70.6%
round  1 | loss 0.6894 | val Acc 48.16 F1 10.99 AUC 47.04 | 54s
round  2 | loss 0.6884 | val Acc 48.08 F1 30.14 AUC 49.11 | 54s
round  3 | loss 0.6875 | val Acc 51.84 F1 54.19 AUC 51.74 | 54s
round  4 | loss 0.6861 | val Acc 52.00 F1 64.81 AUC 55.25 | 54s
round  5 | loss 0.6842 | val Acc 50.76 F1 66.79 AUC 60.17 | 53s
round  6 | loss 0.6811 | val Acc 50.12 F1 66.70 AUC 65.84 | 53s
round  7 | loss 0.6780 | val Acc 50.08 F1 66.70 AUC 70.50 | 54s
round  8 | loss 0.6749 | val Acc 50.00 F1 66.67 AUC 74.34 | 54s
round  9 | loss 0.6718 | val Acc 50.00 F1 66.67 AUC 78.06 | 54s
round 10 | loss 0.6657 | val Acc 50.00 F1 66.67 AUC 81.22 | 54s
round 11 | loss 0.6569 | val Acc 50.04 F1 66.68 AUC 83.62 | 53s
round 12 | loss 0.6461 | val Acc 50.08 F1 66.70 AUC 85.19 | 52s
round 13 | loss 0.6324 | val Acc 51.08 F1 67.11 AUC 86.51 | 53s
round 14

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=2, seed=2)] val F1 99.28 | test1 Acc/F1 99.8/99.8 | OOD Acc/F1/AUC 82.3/82.2/91.8


In [ ]:
# Ô 38 — FedProx (μ=0.1) với Light-MFND, α=0.1 (so với FedAvg α=0.1: OOD F1 57.97, AUC 83.55, 3 seed)
run_fed_light(0.1, MU=0.1)

client 0: n= 1608 | fake=  2.7%
client 1: n=  844 | fake=  0.0%
client 2: n= 5953 | fake= 78.1%
client 3: n= 1271 | fake=  0.0%
client 4: n=  324 | fake= 93.8%
round  1 | loss 0.7191 | val Acc 50.00 F1 66.67 AUC 73.45 | 93s
round  2 | loss 0.7266 | val Acc 50.00 F1 66.67 AUC 89.69 | 92s
round  3 | loss 0.7399 | val Acc 50.00 F1 66.67 AUC 96.82 | 88s
round  4 | loss 0.7383 | val Acc 50.00 F1 66.67 AUC 98.76 | 88s
round  5 | loss 0.6795 | val Acc 59.28 F1 71.06 AUC 99.32 | 91s
round  6 | loss 0.5538 | val Acc 96.60 F1 96.64 AUC 99.45 | 82s
round  7 | loss 0.3726 | val Acc 97.40 F1 97.36 AUC 99.63 | 81s
round  8 | loss 0.2004 | val Acc 97.84 F1 97.81 AUC 99.75 | 82s
round  9 | loss 0.1193 | val Acc 97.92 F1 97.89 AUC 99.82 | 87s
round 10 | loss 0.0909 | val Acc 98.52 F1 98.51 AUC 99.87 | 87s
round 11 | loss 0.0781 | val Acc 98.56 F1 98.55 AUC 99.89 | 85s
round 12 | loss 0.0723 | val Acc 98.72 F1 98.71 AUC 99.91 | 85s
round 13 | loss 0.0667 | val Acc 98.88 F1 98.87 AUC 99.92 | 83s
round 14

'/content/drive/MyDrive/NLP_project/fed/light_a0.1_mu0.1_split0_s0_K5_best.pt'

In [ ]:
# Ô 40 — Đối chứng ensemble: trung bình xác suất của HAI mô hình Light-MFND đầy đủ (khác seed)
# Mục đích: kiểm tra lợi thế của kết hợp muộn có phải chỉ do "gộp hai mô hình" hay không
import os, gc, numpy as np, torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
if "m" not in globals(): m = build()

@torch.no_grad()
def probs(mm, dset, bs=64):
    mm.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            _, lg = mm(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def ensemble_light(paths, TAG):
    for p in paths: assert os.path.exists(p), f"Thiếu file: {p}"
    SPL = ["validation"] + TESTS; P = []
    for p in paths:
        set_state(m, INIT); set_state(m, torch.load(p)); P.append({s: probs(m, enc[s]) for s in SPL})
    res = {}
    for s in SPL:
        y = P[0][s][1]; assert all((x[s][1] == y).all() for x in P)
        pr = np.mean([x[s][0] for x in P], 0); pred = (pr > 0.5).astype(int)
        res["val" if s == "validation" else s] = dict(acc=float(accuracy_score(y, pred) * 100),
                                                       f1=float(f1_score(y, pred) * 100), auc=float(roc_auc_score(y, pr) * 100))
    res["OOD"] = {k: float(np.mean([res[s][k] for s in TESTS[1:]])) for k in ("acc", "f1", "auc")}
    print(f"[{TAG}] val F1 {res['val']['f1']:.2f} | test1 Acc/F1 {res[TESTS[0]]['acc']:.1f}/{res[TESTS[0]]['f1']:.1f} | "
          f"OOD Acc/F1/AUC {res['OOD']['acc']:.1f}/{res['OOD']['f1']:.1f}/{res['OOD']['auc']:.1f}")
    save_result(TAG, res)

# (1) Tập trung: Light-MFND seed 1 + seed 2   — so với kết hợp muộn tập trung: OOD AUC 91.5 (seed 1), 90.9 (seed 2)
ensemble_light([f"{FED}/cen_s1_best.pt", f"{FED}/cen_s2_best.pt"],
               "Đối chứng ensemble: TB 2 Light-MFND tập trung (seed 1 + 2)")
# (2) Học liên kết: FedAvg Light-MFND α=0.5 split 0, seed 0 + seed 1 — so với kết hợp muộn FL split 0: 86.9 (20 vòng), 87.9 (40 vòng)
ensemble_light([f"{FED}/fed_FedAvg-LoRA_(a=0.5,_mu=0.0)_best.pt", f"{FED}/fed_FedAvg-LoRA_(a=0.5,_mu=0.0,_seed=1)_best.pt"],
               "Đối chứng ensemble: TB 2 FedAvg Light-MFND α=0.5 split 0 (seed 0 + 1)")

[Đối chứng ensemble: TB 2 Light-MFND tập trung (seed 1 + 2)] val F1 99.44 | test1 Acc/F1 100.0/100.0 | OOD Acc/F1/AUC 71.0/67.2/80.1
[Đối chứng ensemble: TB 2 FedAvg Light-MFND α=0.5 split 0 (seed 0 + 1)] val F1 99.16 | test1 Acc/F1 99.6/99.6 | OOD Acc/F1/AUC 74.0/68.0/86.3


In [ ]:
# Ô 41a — Kiểm tra dữ liệu trùng giữa các tập test (trước khi dùng test2/test3 để huấn luyện)
import numpy as np, hashlib
from collections import Counter
def rows(sp, what): return np.load(f"{FED}/npy/{sp}_{what}.npy", mmap_mode="r")
h = lambda x: hashlib.md5(np.ascontiguousarray(x).tobytes()).hexdigest()
for a, b in [("test2_bbc_dalle", "test4_bbc_sdxl"), ("test3_cnn_dalle", "test5_cnn_sdxl"),
             ("test2_bbc_dalle", "test5_cnn_sdxl"), ("test3_cnn_dalle", "test4_bbc_sdxl")]:
    la, lb = rows(a, "lab"), rows(b, "lab")
    ta = {h(x): l for x, l in zip(rows(a, "ids"), la)}
    txt = Counter(("real" if lb[j] == 0 else "fake") for j, x in enumerate(rows(b, "ids")) if h(x) in ta)
    pa = {h(x) for x in rows(a, "pix")}
    img = Counter(("real" if lb[j] == 0 else "fake") for j, x in enumerate(rows(b, "pix")) if h(x) in pa)
    print(f"{a} vs {b}: chú thích trùng {dict(txt)} | ảnh trùng {dict(img)} (trên {len(lb)} mẫu của {b})")

test2_bbc_dalle vs test4_bbc_sdxl: chú thích trùng {'fake': 209, 'real': 209} | ảnh trùng {'real': 209} (trên 500 mẫu của test4_bbc_sdxl)
test3_cnn_dalle vs test5_cnn_sdxl: chú thích trùng {'fake': 211, 'real': 211} | ảnh trùng {'real': 211} (trên 500 mẫu của test5_cnn_sdxl)
test2_bbc_dalle vs test5_cnn_sdxl: chú thích trùng {} | ảnh trùng {} (trên 500 mẫu của test5_cnn_sdxl)
test3_cnn_dalle vs test4_bbc_sdxl: chú thích trùng {} | ảnh trùng {} (trên 500 mẫu của test4_bbc_sdxl)


## 9. Summary tables (read `results.json` only)

In [ ]:
# Ô 30 — Bảng tổng hợp cuối cùng (chỉ đọc results.json, chạy được trên CPU)
import os, json, numpy as np, pandas as pd
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
FED = "/content/drive/MyDrive/NLP_project/fed"
R = json.load(open(f"{FED}/results.json"))

LORA, FULL, IMG, TXT = 8.27, 611.4, 1.18, 0.79      # MB mỗi lượt gửi
s3 = lambda base: [base, base.replace(")", ", seed=1)"), base.replace(")", ", seed=2)")]
GROUPS = [   # (nhóm, tên, các khóa trong results.json, MB mỗi lượt gửi; None = không truyền thông)
 ("Tập trung", "Light-MFND (5 epoch)", ["Centralized (LoRA)", "Centralized (LoRA, seed=1)", "Centralized (LoRA, seed=2)"], None),
 ("Tập trung", "Light-MFND (20 epoch, chọn AUC)", ["Centralized (LoRA, 20ep, chọn AUC, seed=0)"], None),
 ("Tập trung", "+ tăng cường ảnh", ["Centralized + tăng cường ảnh (seed=1)"], None),
 ("Tập trung", "Chỉ ảnh", ["Centralized chỉ ảnh (seed=1)", "Centralized chỉ ảnh (seed=2)"], None),
 ("Tập trung", "Chỉ chữ", ["Centralized chỉ chữ (seed=1)", "Centralized chỉ chữ (seed=2)"], None),
 ("Tập trung", "Kết hợp muộn ảnh + chữ", ["Kết hợp muộn: TB xác suất ảnh + chữ (seed=1)", "Kết hợp muộn: TB xác suất ảnh + chữ (seed=2)"], None),
 ("Học liên kết", "FedAvg-LoRA α=100", s3("FedAvg-LoRA (a=100, mu=0.0)"), LORA),
 ("Học liên kết", "FedAvg-LoRA α=0.5 (split 0)", s3("FedAvg-LoRA (a=0.5, mu=0.0)"), LORA),
 ("Học liên kết", "FedAvg-LoRA α=0.5 (split 1)", ["FedAvg-LoRA (a=0.5, mu=0.0, split=1, seed=0)"], LORA),
 ("Học liên kết", "FedAvg-LoRA α=0.5 (split 2)", ["FedAvg-LoRA (a=0.5, mu=0.0, split=2, seed=0)"], LORA),
 ("Học liên kết", "FedAvg-LoRA α=0.1", s3("FedAvg-LoRA (a=0.1, mu=0.0)"), LORA),
 ("Học liên kết", "FedAvg full fine-tune α=0.5", ["FedAvg-FullFT (a=0.5)"], FULL),
 ("Học liên kết", "FedAvg chỉ ảnh α=0.5 (split 0)", ["FedAvg-LoRA chỉ ảnh (a=0.5, split=0, seed=0)"], IMG),
 ("Học liên kết", "FedAvg chỉ ảnh α=0.5 (split 2)", ["FedAvg-LoRA chỉ ảnh (a=0.5, split=2, seed=0)"], IMG),
 ("Học liên kết", "FedAvg chỉ chữ α=0.5 (split 0)", ["FedAvg-LoRA chỉ chữ (a=0.5, split=0, seed=0)"], TXT),
 ("Học liên kết", "FedAvg chỉ chữ α=0.5 (split 2)", ["FedAvg-LoRA chỉ chữ (a=0.5, split=2, seed=0)"], TXT),
 ("Học liên kết", "FedAvg kết hợp muộn α=0.5 (split 0)", ["FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=0, seed=0)"], IMG + TXT),
 ("Học liên kết", "FedAvg kết hợp muộn α=0.5 (split 2)", ["FedAvg kết hợp muộn ảnh + chữ (a=0.5, split=2, seed=0)"], IMG + TXT),
 ("Tự học riêng", "Local-only α=0.5 (TB 5 client)", ["Local-only (a=0.5)"], None),
 ("Tự học riêng", "Local-only α=0.1 (TB 5 client)", ["Local-only (a=0.1)"], None),
]
MET = [("test1 F1", lambda r: r["test1_nyt_mj"]["f1"]), ("OOD Acc", lambda r: r["OOD"]["acc"]),
       ("OOD F1", lambda r: r["OOD"]["f1"]), ("OOD AUC", lambda r: r["OOD"]["auc"])]
rows, missing = [], []
for grp, name, keys, mb in GROUPS:
    ks = [k for k in keys if k in R]; missing += [k for k in keys if k not in R]
    if not ks: continue
    row = {"Nhóm": grp, "Phương pháp": name, "Số lần": len(ks)}
    for mname, f in MET:
        v = [f(R[k]) for k in ks]
        row[mname] = f"{np.mean(v):.2f} ± {np.std(v, ddof=1):.2f}" if len(v) > 1 else f"{v[0]:.2f}"
    row["MB/lượt gửi"] = mb if mb else "–"
    row["Truyền thông 20 vòng (GB, ước tính)"] = round(2 * 20 * 5 * mb / 1e3, 2) if mb else "–"
    rows.append(row)
T = pd.DataFrame(rows)
pd.set_option("display.width", 250); print(T.to_string(index=False))
T.to_csv(f"{FED}/bang_tong_hop.csv", index=False, encoding="utf-8-sig")
print("\nĐã lưu:", f"{FED}/bang_tong_hop.csv")
print("Chưa có kết quả (bỏ qua):", missing)

        Nhóm                         Phương pháp  Số lần      test1 F1      OOD Acc       OOD F1      OOD AUC MB/lượt gửi Truyền thông 20 vòng (GB, ước tính)
   Tập trung                Light-MFND (5 epoch)       3 100.00 ± 0.00 71.33 ± 1.36 68.11 ± 2.22 80.61 ± 1.09           –                                   –
   Tập trung     Light-MFND (20 epoch, chọn AUC)       1        100.00        70.00        62.72        81.09           –                                   –
   Tập trung                    + tăng cường ảnh       1         99.60        63.20        54.39        71.92           –                                   –
   Tập trung                             Chỉ ảnh       2  99.70 ± 0.14 76.85 ± 0.92 74.21 ± 2.11 86.51 ± 0.22           –                                   –
   Tập trung                             Chỉ chữ       2  88.69 ± 0.35 65.95 ± 0.71 74.41 ± 0.38 85.99 ± 0.51           –                                   –
   Tập trung              Kết hợp muộn ảnh + chữ    

In [ ]:
# Ô 31 — In lại kết quả mọi lần chạy từ results.json
# (các ô chạy nhiều lần với tham số khác nhau chỉ giữ phần in ra của lần cuối; ô này in lại tất cả)
import json
R = json.load(open(f"{FED}/results.json"))
for name, r in R.items():
    print("=" * 100); print(name)
    for x in r.get("log", []):
        if "round" in x:
            print(f"  round {x['round']:2d} | loss {x['loss']:.4f} | val F1 {x['val_f1']:.2f}"
                  + (f" AUC {x['val_auc']:.2f}" if "val_auc" in x else ""))
        elif "epoch" in x:
            print(f"  epoch {x['epoch']:2d} | loss {x['loss']:.4f} | val F1 {x['val_f1']:.2f} AUC {x['val_auc']:.2f}")
    t1, ood = r.get("test1_nyt_mj", {}), r.get("OOD", {})
    if t1 and ood:
        print(f"  => val F1 {r['val']['f1']:.2f} | test1 Acc/F1 {t1['acc']:.1f}/{t1['f1']:.1f} | "
              f"OOD Acc/F1/AUC {ood['acc']:.1f}/{ood['f1']:.1f}/{ood['auc']:.1f}")

Local-only (a=0.5)
  => val F1 64.50 | test1 Acc/F1 69.1/65.8 | OOD Acc/F1/AUC 57.4/56.5/66.4
Centralized (LoRA)
  => val F1 99.44 | test1 Acc/F1 100.0/100.0 | OOD Acc/F1/AUC 72.4/70.3/81.7
FedAvg-LoRA (a=100, mu=0.0)
  round  1 | loss 0.7256 | val F1 0.16 AUC 88.85
  round  2 | loss 0.7197 | val F1 78.90 AUC 95.98
  round  3 | loss 0.7061 | val F1 90.32 AUC 98.08
  round  4 | loss 0.6721 | val F1 90.60 AUC 98.90
  round  5 | loss 0.5957 | val F1 93.68 AUC 99.40
  round  6 | loss 0.4590 | val F1 96.73 AUC 99.67
  round  7 | loss 0.2956 | val F1 97.65 AUC 99.79
  round  8 | loss 0.1766 | val F1 98.20 AUC 99.85
  round  9 | loss 0.1184 | val F1 98.45 AUC 99.88
  round 10 | loss 0.0934 | val F1 98.72 AUC 99.90
  round 11 | loss 0.0792 | val F1 98.88 AUC 99.92
  round 12 | loss 0.0685 | val F1 99.00 AUC 99.93
  round 13 | loss 0.0630 | val F1 99.12 AUC 99.94
  round 14 | loss 0.0560 | val F1 98.89 AUC 99.95
  round 15 | loss 0.0511 | val F1 99.20 AUC 99.95
  round 16 | loss 0.0472 | val F1

## Finish
Flush all files to Google Drive.

In [ ]:
# Ô cuối — Đảm bảo mọi file đã ghi lên Drive trước khi đóng Colab
from google.colab import drive
drive.flush_and_unmount(); print("Đã đồng bộ và ngắt Drive")

Đã đồng bộ và ngắt Drive
